# Assignment 04, Thực nghiệm gốc 1: Độ tin cậy thống kê của các bảng đối chuẩn

**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên:** PGS.TS Trần Đình Quế
**Học phần:** Phát triển các Hệ thống Thông minh, Học kỳ 1 năm học 2026 – 2027

## Mục tiêu của notebook

Mọi bảng đối chuẩn ở các chương trước đều xếp cạnh nhau ba con số accuracy của ba khung thư viện,
rồi ngầm mời người đọc kết luận rằng khung có con số lớn hơn thì tốt hơn. Chênh lệch điển hình
giữa hai khung chỉ vào khoảng 0,2 đến 0,3 điểm phần trăm. Câu hỏi mà toàn bộ báo cáo cho tới lúc
này chưa đặt ra là: **một chênh lệch cỡ đó có thật sự tồn tại, hay chỉ là nhiễu sinh ra từ một
hạt giống khởi tạo duy nhất?**

Notebook này trả lời bằng ba công cụ thống kê chứ không bằng khẳng định:

1. **Phương sai đa hạt giống.** Huấn luyện lại mỗi cấu hình với năm hạt giống
   `seed ∈ {42, 43, 44, 45, 46}` và báo cáo `trung bình ± độ lệch chuẩn` thay cho một con số trần.
2. **Kiểm định McNemar** cho từng cặp khung trên **cùng một** tập kiểm thử, kèm bảng bất đồng 2×2,
   giá trị thống kê, p-value và kết luận ở mức ý nghĩa $\alpha = 0{,}05$.
3. **Khoảng tin cậy Wilson 95%** cho từng giá trị accuracy, để thấy khoảng giao nhau tới mức nào.

Đây là đóng góp riêng của báo cáo. Một trong hai bài tham khảo liệt kê chính sự thiếu vắng phân
tích này vào phần hạn chế, nên phần dưới đây lấp đúng chỗ trống đó.

## Kỳ vọng nêu trước khi nhìn thấy kết quả

Báo cáo ghi rõ kỳ vọng tại đây, **trước** khi chạy bất kỳ phép kiểm nào, để phần diễn giải phía sau
không thể được viết lại cho vừa với số liệu.

Luận điểm trung tâm của toàn bộ Assignment 04 là: **nền toán học quyết định kết quả, khung thư viện
chỉ quyết định cách gõ ra công thức đó.** Ba hiện thực NumPy, PyTorch và TensorFlow cùng tối ưu một
hàm mất mát, trên cùng một kiến trúc, cùng một phép chia dữ liệu, cùng một bộ siêu tham số. Nếu
luận điểm ấy đúng thì:

- Độ lệch chuẩn của accuracy qua năm hạt giống phải **cùng cỡ hoặc lớn hơn** khoảng cách trung bình
  giữa hai khung. Nói cách khác, đổi hạt giống gây ra biến động ngang với đổi khung.
- **Phần lớn các cặp khung sẽ KHÔNG khác biệt có ý nghĩa thống kê** trong kiểm định McNemar.
- Các khoảng tin cậy Wilson của ba khung trên cùng một miền sẽ **giao nhau rộng**.

Cần nói thẳng một điều về cách đọc kết quả: nếu quả thật phần lớn các cặp không khác biệt, đó
**không phải một thất bại của thực nghiệm mà là bằng chứng khẳng định luận điểm**. Một kết quả
"không bác bỏ được giả thuyết không" ở đây mang thông tin mạnh, vì nó phủ định trực tiếp cách đọc
bảng đối chuẩn theo kiểu xếp hạng khung thư viện.

Ngược lại, nếu có cặp nào khác biệt có ý nghĩa thật, báo cáo sẽ nói thẳng ra, giữ nguyên số liệu và
đi tìm chi tiết hiện thực nào gây ra chênh lệch đó, tuyệt đối không chỉnh thực nghiệm cho khớp câu
chuyện đã kể trước.

In [ ]:
import os, sys, json, time, math, re, gc, copy, collections, warnings

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

import scipy
from scipy import stats

import torch
import torch.nn as nn

import tensorflow as tf
import keras
from keras import layers

from IPython.display import Image, display

# ----- Cấu hình chung -----
SEEDS = [42, 43, 44, 45, 46]          # danh sách hạt giống đầy đủ của thực nghiệm

# Số hạt giống theo từng miền. Hai miền có biên độ chênh lệch sít sao nhất giữ đủ năm hạt
# giống; hai miền còn lại dùng ba, vì máy đang chạy song song nhiều notebook nặng và một
# lần chạy trước đó đã đổ vì hết bộ nhớ hệ thống. Tệp JSON ghi lại đúng danh sách hạt
# giống thực tế của từng miền để báo cáo phát biểu trung thực.
SEEDS_BY_DOMAIN = {
    "customer_comments": [42, 43, 44, 45, 46],
    "diabetes":          [42, 43, 44, 45, 46],
    "house_price":       [42, 43, 44],
    "mnist":             [42, 43, 44],
}
SPLIT_SEED = 42                       # hạt giống chia dữ liệu, CỐ ĐỊNH ở mọi lần chạy
ALPHA = 0.05                          # mức ý nghĩa của kiểm định McNemar
Z95 = float(stats.norm.ppf(0.975))    # phân vị chuẩn hai phía cho khoảng tin cậy 95%

np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)
torch.use_deterministic_algorithms(False)
torch.set_num_threads(4)              # 32 luồng làm PyTorch chậm gấp 16 lần trên máy này

plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

BASE_DIR = ".."
DATA_COMMENTS = "../../customer_comments/data/womens_ecommerce_reviews.csv"
DATA_DIABETES = "../../diabetes/data/diabetes_prediction_dataset.csv"
DATA_HOUSE    = "../../house_price/data/usa_real_estate_150k.csv"
DATA_MNIST    = "../../mnist/data/mnist.npz"
MNIST_MODELS  = "../../mnist/models"

FIG_DIR = "../reports/figures"
REP_DIR = "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

# ----- Quy tắc thiết bị theo CONTRACT.md Mục 1 -----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Phai chay tren GPU; kiem tra lai venv"

print("Python     :", sys.version.split()[0])
print("NumPy      :", np.__version__)
print("pandas     :", pd.__version__)
print("SciPy      :", scipy.__version__)
print("PyTorch    :", torch.__version__)
print("TensorFlow :", tf.__version__, "| Keras:", keras.__version__)
print()
print("Thiết bị PyTorch    :", DEVICE, "|", torch.cuda.get_device_name(0))
print("Bộ nhớ GPU khả dụng :",
      f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("Số luồng CPU PyTorch:", torch.get_num_threads())
print("Thiết bị TensorFlow : CPU (TF bỏ hỗ trợ GPU native trên Windows từ bản 2.11)")
print()
print("Danh sách hạt giống theo từng miền:")
for _dom, _sds in SEEDS_BY_DOMAIN.items():
    print(f"  {_dom:<20}{str(_sds):<24}({len(_sds)} lần chạy mỗi khung)")
print("Hạt giống chia dữ liệu (cố định):", SPLIT_SEED)
print(f"Mức ý nghĩa alpha   : {ALPHA}   |   phân vị z_(0,975) = {Z95:.6f}")

In [ ]:
try:
    from statsmodels.stats.contingency_tables import mcnemar as sm_mcnemar
    HAS_STATSMODELS = True
    import statsmodels
    print("statsmodels", statsmodels.__version__, "khả dụng: dùng statsmodels.stats."
          "contingency_tables.mcnemar")
except ImportError:
    HAS_STATSMODELS = False
    sm_mcnemar = None
    print("Không tìm thấy statsmodels trong môi trường.")
    print("Báo cáo chuyển sang bản CHÍNH XÁC tự cài bằng scipy.stats.binomtest(b, b+c, 0.5),")
    print("đúng như phương án dự phòng mà CONTRACT.md Mục 8.1 quy định.")

MCNEMAR_BACKEND = "statsmodels" if HAS_STATSMODELS else "scipy.stats.binomtest"
print()
print("Bộ cài kiểm định McNemar được dùng:", MCNEMAR_BACKEND)

## Thiết kế thực nghiệm: cái gì thay đổi và cái gì không

Một thực nghiệm đa hạt giống chỉ có nghĩa khi **đúng một** nguồn ngẫu nhiên được phép thay đổi.
Bảng dưới đây ghi rõ ranh giới đó.

| Thành phần | Trạng thái | Lý do |
|---|---|---|
| Kiến trúc mạng | Cố định theo CONTRACT.md Mục 4 | Đổi kiến trúc thì không còn so sánh khung nữa |
| Siêu tham số (epoch, batch, learning rate) | Cố định theo notebook gốc từng miền | Giữ nguyên để kết quả nối được với các chương trước |
| Phép chia train/validation/test | Cố định `random_state = 42` | Kiểm định McNemar **bắt buộc** hai mô hình chạy trên cùng các mẫu |
| Khởi tạo trọng số | **Thay đổi theo hạt giống** | Đây chính là nguồn nhiễu cần đo |
| Thứ tự xáo trộn lô huấn luyện | **Thay đổi theo hạt giống** | Cùng một nguồn ngẫu nhiên với khởi tạo |

Việc giữ nguyên phép chia dữ liệu là bắt buộc chứ không phải tùy chọn. Nếu tập kiểm thử cũng đổi
theo hạt giống thì hai mô hình sẽ được chấm trên hai tập mẫu khác nhau, và bảng bất đồng 2×2 của
McNemar mất hoàn toàn ý nghĩa. Vì vậy thực nghiệm này đo **nhiễu khởi tạo và nhiễu xáo trộn**, là
đúng loại nhiễu mà một bảng đối chuẩn một lần chạy đã bỏ qua.

Bốn miền được chọn vì đủ rẻ để lặp lại nhiều lần trong ngân sách thời gian:

| Miền | Bài toán | Chỉ số | Số khung lặp lại | Số hạt giống |
|---|---|---|---|---|
| `customer_comments` | Phân loại nhị phân văn bản | accuracy | 3 (NumPy, PyTorch, TensorFlow) | 5 |
| `diabetes` | Phân loại nhị phân bảng | accuracy | 3 (NumPy, PyTorch, TensorFlow) | 5 |
| `house_price` | Hồi quy bảng | R² trên thang log | 3 (NumPy, PyTorch, TensorFlow) | 3 |
| `mnist` | Phân loại 10 lớp ảnh | accuracy | 2 (PyTorch, TensorFlow) | 3 |

**Vì sao số hạt giống không đồng đều, và đây là một hạn chế cần nói thẳng.** Hai miền
`customer_comments` và `diabetes` giữ đủ năm hạt giống vì đó chính là nơi biên độ chênh lệch giữa
các khung sít sao nhất trong toàn báo cáo, lần lượt khoảng 0,50 và 0,26 điểm phần trăm, nên phương
sai ở đó mang nhiều thông tin nhất. Hai miền `house_price` và `mnist` dùng ba hạt giống, vì máy
đang chạy song song nhiều notebook nặng của các phần khác trong Assignment 04 và một lần chạy
trước đó của chính notebook này đã đổ giữa chừng với `MemoryError` ở cấp hệ thống. Ước lượng độ
lệch chuẩn từ ba quan sát kém chắc hơn từ năm quan sát, và bảng kết quả của hai miền đó phải được
đọc với lưu ý ấy. Danh sách hạt giống thực tế của từng miền được ghi vào tệp JSON để báo cáo phát
biểu đúng con số, không phải con số dự định ban đầu.

Hai mô hình 2D thuần NumPy của `mnist` bị loại khỏi phần lặp lại một cách có chủ ý. Một lần chạy
đã tốn 120 giây và 148 giây, nên năm lần lặp cho cả hai biến thể sẽ chiếm phần lớn ngân sách thời
gian mà không đổi lấy thông tin gì mới, vì so sánh trọng tâm của notebook này là giữa **các khung
thư viện** chứ không phải giữa hai cách khởi tạo của cùng một hiện thực NumPy.

Cần ghi nhận trước một sai lệch có thật so với các notebook gốc: notebook `mnist` chương trước chạy
PyTorch trên **CPU**, còn CONTRACT.md Mục 1 bắt buộc mọi notebook PyTorch phải chạy trên **GPU**.
Notebook này tuân thủ hợp đồng, nên con số của hạt giống 42 ở miền `mnist` sẽ **không trùng khít**
với con số 0,9898 đã công bố. Kiến trúc, phép chia, siêu tham số và thứ tự lô đều giữ nguyên; chỉ
thiết bị tính toán là khác, và khác biệt đó đủ để đổi thứ tự cộng dồn dấu phẩy động. Báo cáo ghi
nhận điều này thay vì che đi.

# PHẦN A. Phương sai đa hạt giống

## A.1. Miền `customer_comments`

Kiến trúc theo CONTRACT.md Mục 4: `tokens(50) → Embedding(5000, 100) → Conv1D(32, K=3) → ReLU
→ GlobalMaxPool1D → Dense(1, Sigmoid)`, tổng 509.665 tham số ở cả ba khung. Siêu tham số giữ
nguyên theo ba notebook gốc: 12 epoch, lô 64, Adam với learning rate $10^{-3}$.

Tiền xử lý chỉ chạy **một lần** và dùng chung cho toàn bộ năm hạt giống, vì tiền xử lý không phụ
thuộc hạt giống huấn luyện: từ điển được đếm trên nhánh train của phép chia cố định.

In [ ]:
VOCAB_SIZE, MAX_LEN, EMBED_DIM = 5000, 50, 100
N_FILTERS, KERNEL = 32, 3

_df = pd.read_csv(DATA_COMMENTS)
CMT_N_RAW = len(_df)
_df = _df.dropna(subset=["Review Text", "Recommended IND"]).reset_index(drop=True)
CMT_N_CLEAN = len(_df)


def cmt_tokenize(text):
    """Hạ chữ thường rồi lấy mọi cụm ký tự chữ cái liên tiếp, đúng quy tắc hợp đồng."""
    return re.findall(r"[a-zA-Z]+", text.lower())


cmt_tokens = [cmt_tokenize(t) for t in _df["Review Text"].astype(str)]
cmt_labels = _df["Recommended IND"].values.astype(np.int64)

_idx_all = np.arange(CMT_N_CLEAN)
_idx_tmp, _idx_te = train_test_split(_idx_all, test_size=0.15,
                                     stratify=cmt_labels, random_state=SPLIT_SEED)
_idx_tr, _idx_va = train_test_split(_idx_tmp, test_size=0.1765,
                                    stratify=cmt_labels[_idx_tmp], random_state=SPLIT_SEED)

_counter = collections.Counter(w for i in _idx_tr for w in cmt_tokens[i])
cmt_word2id = {"<PAD>": 0, "<UNK>": 1}
for _w, _ in _counter.most_common(VOCAB_SIZE - 2):
    cmt_word2id[_w] = len(cmt_word2id)


def cmt_encode(token_lists):
    """Ánh xạ token sang chỉ số, cắt ở MAX_LEN và đệm 0 ở phía sau."""
    X = np.zeros((len(token_lists), MAX_LEN), dtype=np.int64)
    for i, toks in enumerate(token_lists):
        ids = [cmt_word2id.get(w, 1) for w in toks[:MAX_LEN]]
        X[i, :len(ids)] = ids
    return X


CMT_Xtr = cmt_encode([cmt_tokens[i] for i in _idx_tr])
CMT_Xva = cmt_encode([cmt_tokens[i] for i in _idx_va])
CMT_Xte = cmt_encode([cmt_tokens[i] for i in _idx_te])
CMT_ytr = cmt_labels[_idx_tr].astype(np.float64)
CMT_yva = cmt_labels[_idx_va].astype(np.float64)
CMT_yte = cmt_labels[_idx_te].astype(np.float64)
CMT_N_TEST = len(CMT_yte)

# Giải phóng các cấu trúc trung gian lớn. Máy đang chạy song song nhiều notebook nặng,
# và một lần chạy trước đó đã đổ vì hết bộ nhớ hệ thống.
del _df, cmt_tokens, _counter
gc.collect()

print(f"Thô -> sạch            : {CMT_N_RAW:,} -> {CMT_N_CLEAN:,} review")
print(f"Chia tập               : train {len(CMT_ytr):,} / val {len(CMT_yva):,} "
      f"/ test {CMT_N_TEST:,}")
print(f"Tỉ lệ nhãn 1 mỗi tập   : {CMT_ytr.mean():.4f} / {CMT_yva.mean():.4f} "
      f"/ {CMT_yte.mean():.4f}")
print(f"Kích thước từ điển     : {len(cmt_word2id):,} (đã gồm <PAD> và <UNK>)")
print(f"Ma trận chỉ số         : {CMT_Xtr.shape}, kiểu {CMT_Xtr.dtype}")
print()
print("Phép chia này trùng khớp với ba notebook gốc của miền customer_comments,")
print("vì cùng dùng random_state =", SPLIT_SEED, "và cùng thứ tự hai lần train_test_split.")

In [ ]:
class TxtEmbedding:
    """Bảng tra cứu vector nhúng: (B, L) số nguyên -> (B, L, D)."""

    def __init__(self, vocab_size, dim, rng, scale=0.05):
        self.W = rng.normal(0.0, scale, size=(vocab_size, dim))
        self.W[0] = 0.0
        self.dW = np.zeros_like(self.W)
        self.idx = None

    def forward(self, idx):
        self.idx = idx
        return self.W[idx]

    def backward(self, d_out):
        self.dW = np.zeros_like(self.W)
        np.add.at(self.dW, self.idx, d_out)
        self.dW[0] = 0.0
        return None

    @property
    def params(self): return [self.W]
    @property
    def grads(self):  return [self.dW]


class TxtConv1D:
    """Tích chập 1 chiều chế độ 'valid': (B, L, C_in) -> (B, L-K+1, C_out)."""

    def __init__(self, c_in, c_out, k, rng):
        fan_in, fan_out = c_in * k, c_out
        limit = np.sqrt(6.0 / (fan_in + fan_out))          # Glorot uniform
        self.W = rng.uniform(-limit, limit, size=(c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.k = k

    def forward(self, x):
        self.in_shape = x.shape
        self.windows = sliding_window_view(x, self.k, axis=1)
        # optimize=True cho phép einsum chọn thứ tự rút gọn tốt hơn và gọi xuống BLAS.
        # Đây thuần là cờ hiệu năng: phép co chỉ số không đổi, kết quả chỉ lệch ở mức
        # tái kết hợp dấu phẩy động (đo được khoảng 1e-14 tương đối). Hai ngăn xếp NumPy
        # của diabetes và house_price vốn đã dùng cờ này.
        return np.einsum("bldk,odk->blo", self.windows, self.W, optimize=True) + self.b

    def backward(self, d_out):
        self.dW = np.einsum("blo,bldk->odk", d_out, self.windows, optimize=True)
        self.db = d_out.sum(axis=(0, 1))
        d_win = np.einsum("blo,odk->bldk", d_out, self.W, optimize=True)
        d_x = np.zeros(self.in_shape)
        L_out = d_win.shape[1]
        for j in range(self.k):
            d_x[:, j:j + L_out, :] += d_win[:, :, :, j]
        return d_x

    @property
    def params(self): return [self.W, self.b]
    @property
    def grads(self):  return [self.dW, self.db]


class TxtReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, d_out):
        return d_out * self.mask

    @property
    def params(self): return []
    @property
    def grads(self):  return []


class TxtGlobalMaxPool1D:
    """(B, L', C) -> (B, C), giữ vị trí thắng để định tuyến gradient."""

    def forward(self, x):
        self.in_shape = x.shape
        self.argmax = x.argmax(axis=1)
        return np.take_along_axis(x, self.argmax[:, None, :], axis=1)[:, 0, :]

    def backward(self, d_out):
        d_x = np.zeros(self.in_shape)
        np.put_along_axis(d_x, self.argmax[:, None, :], d_out[:, None, :], axis=1)
        return d_x

    @property
    def params(self): return []
    @property
    def grads(self):  return []


class TxtDense:
    def __init__(self, n_in, n_out, rng):
        limit = np.sqrt(6.0 / (n_in + n_out))
        self.W = rng.uniform(-limit, limit, size=(n_in, n_out))
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, d_out):
        self.dW = self.x.T @ d_out
        self.db = d_out.sum(axis=0)
        return d_out @ self.W.T

    @property
    def params(self): return [self.W, self.b]
    @property
    def grads(self):  return [self.dW, self.db]


def txt_sigmoid(u):
    """Sigmoid ổn định số học, tránh tràn số khi |u| lớn."""
    out = np.empty_like(u)
    pos, neg = u >= 0, u < 0
    out[pos] = 1.0 / (1.0 + np.exp(-u[pos]))
    ex = np.exp(u[neg])
    out[neg] = ex / (1.0 + ex)
    return out


def txt_bce(y_hat, y):
    eps = 1e-12
    p = np.clip(y_hat, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


class TextCNN:
    """tokens(50) -> Embedding(5000,100) -> Conv1D(32,K=3) -> ReLU
                  -> GlobalMaxPool1D -> Dense(1) -> Sigmoid"""

    def __init__(self, vocab_size, embed_dim, n_filters, kernel, rng):
        self.emb  = TxtEmbedding(vocab_size, embed_dim, rng)
        self.conv = TxtConv1D(embed_dim, n_filters, kernel, rng)
        self.relu = TxtReLU()
        self.pool = TxtGlobalMaxPool1D()
        self.fc   = TxtDense(n_filters, 1, rng)
        self.layers = [self.emb, self.conv, self.relu, self.pool, self.fc]

    def forward(self, x_idx):
        h = self.emb.forward(x_idx)
        h = self.conv.forward(h)
        h = self.relu.forward(h)
        h = self.pool.forward(h)
        u = self.fc.forward(h)
        return txt_sigmoid(u)[:, 0], u[:, 0]

    def backward(self, y_hat, y):
        B = y.shape[0]
        d_u = ((y_hat - y) / B)[:, None]          # đạo hàm ghép của sigmoid và BCE
        d = self.fc.backward(d_u)
        d = self.pool.backward(d)
        d = self.relu.backward(d)
        d = self.conv.backward(d)
        self.emb.backward(d)

    @property
    def params(self): return [p for l in self.layers for p in l.params]
    @property
    def grads(self):  return [g for l in self.layers for g in l.grads]

    def n_params(self): return int(sum(p.size for p in self.params))


class TxtAdam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t
        bc2 = 1.0 - self.b2 ** self.t
        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * (g * g)
            p -= self.lr * (self.m[i] / bc1) / (np.sqrt(self.v[i] / bc2) + self.eps)


_probe = TextCNN(VOCAB_SIZE, EMBED_DIM, N_FILTERS, KERNEL, np.random.default_rng(0))
CMT_N_PARAMS = _probe.n_params()
print("Đã sao chép nguyên vẹn ngăn xếp NumPy của notebook customer_comments/01.")
print("Tên lớp được thêm tiền tố Txt để ba ngăn xếp của bốn miền cùng tồn tại trong một notebook;")
print("phần toán học bên trong giữ nguyên từng dòng.")
print()
print(f"Số tham số TextCNN: {CMT_N_PARAMS:,} (đối chiếu hợp đồng: 509.665)")
assert CMT_N_PARAMS == 509665, "So tham so khong khop voi notebook goc"

In [ ]:
CMT_EPOCHS, CMT_BATCH, CMT_LR = 12, 64, 1e-3


def cmt_eval_numpy(net, X, y, batch=128):
    probs = np.empty(len(X))
    for s in range(0, len(X), batch):
        p, _ = net.forward(X[s:s + batch])
        probs[s:s + batch] = p
    return probs, txt_bce(probs, y), float(((probs > 0.5) == y).mean())


def cmt_train_numpy(seed, verbose=False):
    """Bản NumPy thuần. Hạt giống chi phối cả khởi tạo lẫn thứ tự xáo trộn lô."""
    model = TextCNN(VOCAB_SIZE, EMBED_DIM, N_FILTERS, KERNEL, np.random.default_rng(seed))
    opt = TxtAdam(model.params, lr=CMT_LR)
    train_rng = np.random.default_rng(seed)
    n = len(CMT_Xtr)
    best_vl, best_ep, best_params = np.inf, 0, None
    t0 = time.time()

    for ep in range(1, CMT_EPOCHS + 1):
        perm = train_rng.permutation(n)
        for s in range(0, n, CMT_BATCH):
            bi = perm[s:s + CMT_BATCH]
            p, _ = model.forward(CMT_Xtr[bi])
            model.backward(p, CMT_ytr[bi])
            opt.step(model.grads)
        _, vl, va = cmt_eval_numpy(model, CMT_Xva, CMT_yva)
        if vl < best_vl:
            best_vl, best_ep = vl, ep
            best_params = [p.copy() for p in model.params]
        if verbose:
            print(f"    epoch {ep:2d}/{CMT_EPOCHS} val_loss={vl:.4f} val_acc={va:.4f}")

    for p, bp in zip(model.params, best_params):
        p[...] = bp
    probs, _, _ = cmt_eval_numpy(model, CMT_Xte, CMT_yte)
    return (probs > 0.5).astype(np.int64), float(time.time() - t0), int(best_ep)


class TextCNNTorch(nn.Module):
    """Bản PyTorch của cùng kiến trúc, sao chép từ customer_comments/02."""

    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.conv      = nn.Conv1d(EMBED_DIM, N_FILTERS, kernel_size=KERNEL)
        self.relu      = nn.ReLU()
        self.pool      = nn.AdaptiveMaxPool1d(1)
        self.fc        = nn.Linear(N_FILTERS, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.05)
        with torch.no_grad():
            self.embedding.weight[0].zero_()
        nn.init.xavier_uniform_(self.conv.weight); nn.init.zeros_(self.conv.bias)
        nn.init.xavier_uniform_(self.fc.weight);   nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        e = self.embedding(x).transpose(1, 2)
        a = self.relu(self.conv(e))
        p = self.pool(a).squeeze(-1)
        return self.fc(p).squeeze(-1)


def cmt_train_torch(seed, verbose=False):
    """Bản PyTorch chạy trên GPU. Dữ liệu nằm sẵn trên thiết bị, thứ tự lô do randperm quyết định.

    torch.randperm(n, generator=g) sinh đúng hoán vị mà RandomSampler của DataLoader dùng,
    nên thành phần từng lô trùng khớp với notebook gốc ở cùng hạt giống.
    """
    torch.manual_seed(seed)
    model = TextCNNTorch().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CMT_LR)
    crit = nn.BCEWithLogitsLoss()
    g = torch.Generator(); g.manual_seed(seed)

    Xtr = torch.from_numpy(CMT_Xtr).to(DEVICE)
    ytr = torch.from_numpy(CMT_ytr).float().to(DEVICE)
    Xva = torch.from_numpy(CMT_Xva).to(DEVICE)
    yva = torch.from_numpy(CMT_yva).float().to(DEVICE)
    Xte = torch.from_numpy(CMT_Xte).to(DEVICE)

    @torch.no_grad()
    def ev(X, y):
        model.eval()
        tot, n = 0.0, 0
        for i in range(0, len(X), 1024):
            lg = model(X[i:i + 1024])
            tot += float(crit(lg, y[i:i + 1024])) * len(lg)
            n += len(lg)
        return tot / n

    best_vl, best_ep, best_state = float("inf"), 0, None
    torch.cuda.synchronize(); t0 = time.time()
    for ep in range(1, CMT_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(Xtr), generator=g).to(DEVICE)
        for i in range(0, len(order), CMT_BATCH):
            b = order[i:i + CMT_BATCH]
            opt.zero_grad()
            crit(model(Xtr[b]), ytr[b]).backward()
            opt.step()
        vl = ev(Xva, yva)
        if vl < best_vl:
            best_vl, best_ep = vl, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"    epoch {ep:2d}/{CMT_EPOCHS} val_loss={vl:.4f}")
    torch.cuda.synchronize(); el = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = torch.cat([model(Xte[i:i + 1024]) for i in range(0, len(Xte), 1024)])
        pred = (torch.sigmoid(logits) > 0.5).long().cpu().numpy()
    return pred, float(el), int(best_ep)


class CmtBestWeights(keras.callbacks.Callback):
    """Ghi nhớ trọng số của epoch có val_loss nhỏ nhất, giữ trong bộ nhớ chứ không ghi ra đĩa."""

    def __init__(self):
        super().__init__()
        self.best = float("inf"); self.best_epoch = 0; self.weights = None

    def on_epoch_end(self, epoch, logs=None):
        vl = (logs or {}).get("val_loss")
        if vl is not None and vl < self.best:
            self.best, self.best_epoch = float(vl), epoch + 1
            self.weights = [w.copy() for w in self.model.get_weights()]


def cmt_train_tf(seed, verbose=False):
    """Bản TensorFlow/Keras chạy trên CPU, sao chép từ customer_comments/03."""
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        layers.Input(shape=(MAX_LEN,), dtype="int32"),
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                         embeddings_initializer=keras.initializers.RandomNormal(stddev=0.05)),
        layers.Conv1D(filters=N_FILTERS, kernel_size=KERNEL, padding="valid",
                      activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=CMT_LR),
                  loss="binary_crossentropy", metrics=["accuracy"])
    cb = CmtBestWeights()
    t0 = time.time()
    model.fit(CMT_Xtr, CMT_ytr, validation_data=(CMT_Xva, CMT_yva),
              epochs=CMT_EPOCHS, batch_size=CMT_BATCH, shuffle=True,
              verbose=2 if verbose else 0, callbacks=[cb])
    el = time.time() - t0
    model.set_weights(cb.weights)
    probs = model.predict(CMT_Xte, batch_size=1024, verbose=0).ravel()
    return (probs > 0.5).astype(np.int64), float(el), int(cb.best_epoch)


print("Đã định nghĩa ba hàm huấn luyện cho miền customer_comments:")
print("  cmt_train_numpy(seed) | cmt_train_torch(seed) | cmt_train_tf(seed)")
print(f"Siêu tham số chung: {CMT_EPOCHS} epoch, lô {CMT_BATCH}, Adam lr = {CMT_LR}")
print("Cả ba hàm đều trả về (dự đoán nhãn trên tập test, thời gian huấn luyện, epoch tốt nhất).")

In [ ]:
# Kho chứa kết quả của toàn bộ thực nghiệm. PRED[domain][framework][seed] = mảng nhãn dự đoán.
ACC = {}     # ACC[domain][framework] = giá trị chỉ số theo thứ tự SEEDS_BY_DOMAIN[domain]
PRED = {}    # chỉ dùng cho các miền phân loại
TIMES = {}   # thời gian huấn luyện từng lần chạy


def _register(domain, framework, values, preds=None, times=None):
    ACC.setdefault(domain, {})[framework] = values
    TIMES.setdefault(domain, {})[framework] = times
    if preds is not None:
        PRED.setdefault(domain, {})[framework] = preds


def chay_co_thu_lai(fn, sd, so_lan=4, cho_giay=120):
    """Gọi fn(sd), thử lại nếu hệ điều hành từ chối cấp phát bộ nhớ.

    Máy đang chạy song song nhiều notebook nặng của các phần khác trong Assignment 04.
    Giới hạn commit của Windows có lúc cạn, khiến ngay cả một cấp phát vài MiB cũng hỏng,
    dù bộ nhớ vật lý còn trống. Đây là sự cố nhất thời của môi trường chứ không phải của
    mô hình, nên cách xử lý đúng là dọn rác, chờ, rồi chạy lại đúng hạt giống đó. Việc
    thử lại KHÔNG đổi bất kỳ con số nào, vì hạt giống và siêu tham số giữ nguyên.
    """
    for lan in range(1, so_lan + 1):
        try:
            return fn(sd)
        except (MemoryError, torch.cuda.OutOfMemoryError) as e:
            if lan == so_lan:
                raise
            print(f"    [cấp phát bộ nhớ hỏng ở lần thử {lan}/{so_lan}: "
                  f"{type(e).__name__}] dọn rác và chờ {cho_giay}s rồi chạy lại seed {sd}",
                  flush=True)
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(cho_giay)


def run_multi_seed(domain, framework, fn, y_true, metric_name="accuracy"):
    """Chạy fn(seed) cho từng hạt giống của miền và in nhật ký từng lần chạy."""
    seeds = SEEDS_BY_DOMAIN[domain]
    vals, preds, tms = [], {}, []
    print(f"[{domain} | {framework}] bắt đầu {len(seeds)} lần chạy, hạt giống {seeds}")
    for sd in seeds:
        pred, el, best_ep = chay_co_thu_lai(fn, sd)
        acc = float(accuracy_score(y_true, pred))
        vals.append(acc); preds[sd] = pred; tms.append(el)
        print(f"  seed {sd} | {metric_name} = {acc:.6f} | epoch tốt nhất = {best_ep:2d} "
              f"| {el:7.1f}s")
    m, s = float(np.mean(vals)), float(np.std(vals, ddof=1))
    print(f"  => trung bình {m:.6f} ± {s:.6f}  (biên độ "
          f"{max(vals) - min(vals):.6f}, tổng {sum(tms):.1f}s)")
    _register(domain, framework, vals, preds, tms)
    return vals


print("Đã định nghĩa run_multi_seed và ba kho chứa ACC, PRED, TIMES.")
print("Độ lệch chuẩn dùng ddof=1 (ước lượng không chệch cho mẫu năm quan sát).")

In [ ]:
_t_cmt = time.time()
run_multi_seed("customer_comments", "numpy", cmt_train_numpy, CMT_yte)
print()
run_multi_seed("customer_comments", "pytorch", cmt_train_torch, CMT_yte)
print()
run_multi_seed("customer_comments", "tensorflow", cmt_train_tf, CMT_yte)
print()
print(f"Tổng thời gian miền customer_comments: {time.time() - _t_cmt:.1f}s")

In [ ]:
def bang_da_hat_giong(domain, metric="accuracy"):
    """In bảng trung bình ± độ lệch chuẩn cùng khoảng cách giữa các khung."""
    d = ACC[domain]
    seeds = SEEDS_BY_DOMAIN[domain]
    print(f"{'Khung':<14}" + "".join(f"{s:>12}" for s in seeds) +
          f"{'trung bình':>14}{'độ lệch chuẩn':>16}{'biên độ':>12}")
    print("-" * (14 + 12 * len(seeds) + 14 + 16 + 12))
    for fw, vals in d.items():
        print(f"{fw:<14}" + "".join(f"{v:>12.6f}" for v in vals) +
              f"{np.mean(vals):>14.6f}{np.std(vals, ddof=1):>16.6f}"
              f"{max(vals) - min(vals):>12.6f}")
    print()
    names = list(d.keys())
    print(f"{'Cặp khung':<28}{'|chênh lệch trung bình|':>26}{'độ lệch chuẩn gộp':>20}"
          f"{'tỉ số':>10}")
    print("-" * 84)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = np.array(d[names[i]]), np.array(d[names[j]])
            gap = abs(a.mean() - b.mean())
            pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
            print(f"{names[i] + ' vs ' + names[j]:<28}{gap:>26.6f}{pooled:>20.6f}"
                  f"{gap / pooled if pooled > 0 else float('nan'):>10.3f}")
    print()
    print(f"Chỉ số: {metric}. Tỉ số nhỏ hơn 1 nghĩa là khoảng cách giữa hai khung còn NHỎ HƠN")
    print("mức dao động do đổi hạt giống, tức là không thể coi là khác biệt thật.")


bang_da_hat_giong("customer_comments")

### Diễn giải bảng `customer_comments`

Bảng trên đọc theo hai tầng. Tầng thứ nhất cho biết mỗi khung dao động bao nhiêu khi chỉ đổi hạt
giống. Tầng thứ hai đặt khoảng cách giữa hai khung cạnh độ lệch chuẩn gộp của chính hai khung đó,
và cột tỉ số chính là đại lượng cần đọc: nó trả lời câu hỏi "khoảng cách giữa hai khung bằng mấy
lần mức nhiễu nội tại của chúng".

Một tỉ số nhỏ hơn 1 nói rằng nếu chỉ đổi hạt giống mà không đổi gì khác, người ta đã có thể tạo ra
một chênh lệch lớn hơn chênh lệch đang được quy cho khung thư viện. Trong tình huống đó, xếp hạng
ba khung theo accuracy là xếp hạng nhiễu. Bảng ở Chương 4 của báo cáo ghi NumPy 88,31%, PyTorch
88,05% và TensorFlow 87,81%, tức khoảng cách lớn nhất là 0,50 điểm phần trăm; các con số trong
bảng vừa in cho biết mức dao động thuần do hạt giống có nằm cùng cỡ với 0,50 điểm phần trăm đó hay
không.

## A.2. Miền `diabetes`

Kiến trúc theo CONTRACT.md Mục 4 cho dữ liệu bảng:
`8 → Conv1D(16, K=3, same) → ReLU → Conv1D(16, K=3, same) → ReLU → MaxPool1D(2) → Flatten
→ Dense(8) → ReLU → Dense(1) → Sigmoid`, tổng 1.377 tham số ở cả ba khung. Siêu tham số giữ
nguyên: 20 epoch, lô 256, Adam với learning rate $10^{-3}$.

Miền này đáng chú ý vì nhãn rất mất cân bằng, chỉ khoảng 8,5% mẫu mang nhãn dương. Accuracy do đó
bị kéo lên cao bởi lớp đa số, và chính vì vậy khoảng cách 0,26 điểm phần trăm giữa hai khung càng
cần được kiểm định chứ không thể đọc trực tiếp.

In [ ]:
DIA_FEATURES = ["gender", "age", "hypertension", "heart_disease",
                "smoking_history", "bmi", "HbA1c_level", "blood_glucose_level"]

_raw = pd.read_csv(DATA_DIABETES)
DIA_N_RAW = len(_raw)
_d = _raw.drop_duplicates().copy()
_d = _d[_d["gender"] != "Other"].copy()
_d["gender"] = _d["gender"].map({"Male": 1, "Female": 0})
_d["smoking_history"] = _d["smoking_history"].map({
    "never": 0, "No Info": 0, "former": 1, "not current": 1, "current": 2, "ever": 2})
assert _d[DIA_FEATURES].isna().sum().sum() == 0

_Xall = _d[DIA_FEATURES].values.astype(np.float64)
_yall = _d["diabetes"].values.astype(np.float64)
DIA_N_CLEAN = len(_Xall)

_Xtr_raw, _Xtmp, _ytr, _ytmp = train_test_split(
    _Xall, _yall, test_size=0.30, stratify=_yall, random_state=SPLIT_SEED)
_Xva_raw, _Xte_raw, _yva, _yte = train_test_split(
    _Xtmp, _ytmp, test_size=0.50, stratify=_ytmp, random_state=SPLIT_SEED)

_sc = StandardScaler().fit(_Xtr_raw)
DIA_Xtr = _sc.transform(_Xtr_raw).astype(np.float32)[:, None, :]
DIA_Xva = _sc.transform(_Xva_raw).astype(np.float32)[:, None, :]
DIA_Xte = _sc.transform(_Xte_raw).astype(np.float32)[:, None, :]
DIA_ytr, DIA_yva, DIA_yte = _ytr, _yva, _yte
DIA_N_TEST = len(DIA_yte)

print(f"Thô -> sạch          : {DIA_N_RAW:,} -> {DIA_N_CLEAN:,} bệnh nhân")
print(f"Chia tập             : train {len(DIA_ytr):,} / val {len(DIA_yva):,} "
      f"/ test {DIA_N_TEST:,}")
print(f"Tỉ lệ nhãn dương     : {DIA_ytr.mean():.4f} / {DIA_yva.mean():.4f} / {DIA_yte.mean():.4f}")
print(f"Hình dạng tensor vào : {DIA_Xtr.shape} (N, C_in = 1, L = 8)")
print()
_base = max((DIA_yte == 0).mean(), (DIA_yte == 1).mean())
print(f"Accuracy của bộ phân loại tầm thường (luôn đoán lớp đa số): {_base:.6f}")
print("Mọi accuracy dưới đây phải được đọc trên nền này chứ không phải trên nền 0,5.")

In [ ]:
class TabConv1D:
    """Tích chập 1 chiều chế độ 'same', khởi tạo He Normal."""

    def __init__(self, c_in, c_out, k=3, padding="same", seed=0):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0.0, np.sqrt(2.0 / (c_in * k)), size=(c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.k = k
        self.pad = (k - 1) // 2 if padding == "same" else 0
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        p = self.pad
        Xp = np.pad(X, ((0, 0), (0, 0), (p, p))) if p > 0 else X
        win = sliding_window_view(Xp, self.k, axis=2)
        out = np.einsum("nclk,ock->nol", win, self.W, optimize=True) + self.b[None, :, None]
        self._win = win
        self._shape_p = Xp.shape
        return out

    def backward(self, dout):
        self.dW = np.einsum("nol,nclk->ock", dout, self._win, optimize=True)
        self.db = dout.sum(axis=(0, 2))
        dcol = np.einsum("nol,ock->nclk", dout, self.W, optimize=True)
        dXp = np.zeros(self._shape_p)
        L_out = dout.shape[2]
        for k in range(self.k):
            dXp[:, :, k:k + L_out] += dcol[:, :, :, k]
        p = self.pad
        return dXp[:, :, p:self._shape_p[2] - p] if p > 0 else dXp

    def n_params(self): return self.W.size + self.b.size


class TabReLU:
    def forward(self, X):
        self._mask = X > 0
        return X * self._mask

    def backward(self, dout):
        return dout * self._mask


class TabMaxPool1D:
    def __init__(self, size=2):
        self.size = size

    def forward(self, X):
        N, C, L = X.shape
        s = self.size
        L_out = L // s
        Xr = X[:, :, :L_out * s].reshape(N, C, L_out, s)
        self._arg = Xr.argmax(axis=3)
        self._shape = X.shape
        self._L_out = L_out
        return Xr.max(axis=3)

    def backward(self, dout):
        N, C, L = self._shape
        s, L_out = self.size, self._L_out
        dXr = np.zeros((N, C, L_out, s))
        n, c, l = np.ogrid[:N, :C, :L_out]
        dXr[n, c, l, self._arg] = dout
        dX = np.zeros(self._shape)
        dX[:, :, :L_out * s] = dXr.reshape(N, C, L_out * s)
        return dX


class TabFlatten:
    def forward(self, X):
        self._shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self._shape)


class TabDense:
    def __init__(self, n_in, n_out, seed=0):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out))
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self._X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self._X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

    def n_params(self): return self.W.size + self.b.size


class TabSigmoid:
    def forward(self, X):
        self._out = 1.0 / (1.0 + np.exp(-np.clip(X, -60, 60)))
        return self._out

    def backward(self, dout):
        return dout * self._out * (1.0 - self._out)


def tab_bce_loss(p, y, eps=1e-9):
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


def tab_bce_grad(p, y, eps=1e-9):
    p = np.clip(p, eps, 1.0 - eps)
    return (-(y / p) + (1.0 - y) / (1.0 - p)) / y.size


class TabAdam:
    def __init__(self, layers, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.layers, self.lr = layers, lr
        self.b1, self.b2, self.eps = beta1, beta2, eps
        self.t = 0
        self.m, self.v = {}, {}
        for i, layer in enumerate(layers):
            for name in ("W", "b"):
                self.m[(i, name)] = np.zeros_like(getattr(layer, name))
                self.v[(i, name)] = np.zeros_like(getattr(layer, name))

    def step(self):
        self.t += 1
        for i, layer in enumerate(self.layers):
            for name, g in (("W", layer.dW), ("b", layer.db)):
                key = (i, name)
                self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * g
                self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * g * g
                m_hat = self.m[key] / (1 - self.b1 ** self.t)
                v_hat = self.v[key] / (1 - self.b2 ** self.t)
                setattr(layer, name,
                        getattr(layer, name) - self.lr * m_hat / (np.sqrt(v_hat) + self.eps))


class DiabetesCNN1D:
    """1D CNN thuần NumPy, sao chép từ diabetes/01."""

    def __init__(self, seed=42):
        self.conv1 = TabConv1D(1, 16, k=3, padding="same", seed=seed)
        self.relu1 = TabReLU()
        self.conv2 = TabConv1D(16, 16, k=3, padding="same", seed=seed + 1)
        self.relu2 = TabReLU()
        self.pool  = TabMaxPool1D(2)
        self.flat  = TabFlatten()
        self.fc1   = TabDense(64, 8, seed=seed + 2)
        self.relu3 = TabReLU()
        self.fc2   = TabDense(8, 1, seed=seed + 3)
        self.sig   = TabSigmoid()
        self.layers = [self.conv1, self.relu1, self.conv2, self.relu2, self.pool,
                       self.flat, self.fc1, self.relu3, self.fc2, self.sig]
        self.trainable = [self.conv1, self.conv2, self.fc1, self.fc2]

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def n_params(self): return sum(l.n_params() for l in self.trainable)

    def get_weights(self): return [(l.W.copy(), l.b.copy()) for l in self.trainable]

    def set_weights(self, ws):
        for l, (W, b) in zip(self.trainable, ws):
            l.W, l.b = W.copy(), b.copy()

    def predict_proba(self, X, batch=4096):
        return np.concatenate([self.forward(X[i:i + batch]) for i in range(0, len(X), batch)],
                              axis=0)


DIA_N_PARAMS = DiabetesCNN1D().n_params()
print(f"Số tham số DiabetesCNN1D: {DIA_N_PARAMS:,} (đối chiếu hợp đồng: 1.377)")
assert DIA_N_PARAMS == 1377, "So tham so khong khop"
print("Ngăn xếp NumPy của diabetes/01 đã được sao chép nguyên vẹn, chỉ thêm tiền tố Tab vào tên lớp.")

In [ ]:
DIA_EPOCHS, DIA_BATCH, DIA_LR = 20, 256, 1e-3

_DIA_Xtr64 = DIA_Xtr.astype(np.float64)
_DIA_Xva64 = DIA_Xva.astype(np.float64)
_DIA_Xte64 = DIA_Xte.astype(np.float64)
_DIA_ytr2 = DIA_ytr[:, None]
_DIA_yva2 = DIA_yva[:, None]


def dia_train_numpy(seed, verbose=False):
    model = DiabetesCNN1D(seed=seed)
    opt = TabAdam(model.trainable, lr=DIA_LR)
    rng = np.random.default_rng(seed)
    best_vl, best_ep, best_w = np.inf, 0, model.get_weights()
    t0 = time.time()
    for ep in range(1, DIA_EPOCHS + 1):
        order = rng.permutation(len(_DIA_Xtr64))
        for i in range(0, len(order), DIA_BATCH):
            b = order[i:i + DIA_BATCH]
            p = model.forward(_DIA_Xtr64[b])
            model.backward(tab_bce_grad(p, _DIA_ytr2[b]))
            opt.step()
        vl = tab_bce_loss(model.predict_proba(_DIA_Xva64), _DIA_yva2)
        if vl < best_vl:
            best_vl, best_ep, best_w = vl, ep, model.get_weights()
        if verbose:
            print(f"    epoch {ep:2d}/{DIA_EPOCHS} val_loss={vl:.5f}")
    model.set_weights(best_w)
    p = model.predict_proba(_DIA_Xte64).ravel()
    return (p >= 0.5).astype(np.int64), float(time.time() - t0), int(best_ep)


class DiabetesCNN1DTorch(nn.Module):
    """Bản PyTorch, sao chép từ diabetes/02. Đầu ra là logit."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2), nn.Flatten(),
            nn.Linear(64, 8), nn.ReLU(), nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)


_DIA_Xtr_t = torch.from_numpy(DIA_Xtr).float().to(DEVICE)
_DIA_Xva_t = torch.from_numpy(DIA_Xva).float().to(DEVICE)
_DIA_Xte_t = torch.from_numpy(DIA_Xte).float().to(DEVICE)
_DIA_ytr_t = torch.from_numpy(DIA_ytr[:, None]).float().to(DEVICE)
_DIA_yva_t = torch.from_numpy(DIA_yva[:, None]).float().to(DEVICE)


def dia_train_torch(seed, verbose=False):
    torch.manual_seed(seed)
    model = DiabetesCNN1DTorch().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=DIA_LR)
    crit = nn.BCEWithLogitsLoss()
    g = torch.Generator(); g.manual_seed(seed)
    best_vl, best_ep, best_state = float("inf"), 0, None
    torch.cuda.synchronize(); t0 = time.time()
    for ep in range(1, DIA_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(_DIA_Xtr_t), generator=g).to(DEVICE)
        for i in range(0, len(order), DIA_BATCH):
            b = order[i:i + DIA_BATCH]
            opt.zero_grad()
            crit(model(_DIA_Xtr_t[b]), _DIA_ytr_t[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(crit(model(_DIA_Xva_t), _DIA_yva_t))
        if vl < best_vl:
            best_vl, best_ep = vl, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"    epoch {ep:2d}/{DIA_EPOCHS} val_loss={vl:.5f}")
    torch.cuda.synchronize(); el = time.time() - t0
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(_DIA_Xte_t)).cpu().numpy().ravel()
    return (prob >= 0.5).astype(np.int64), float(el), int(best_ep)


class DiaBestWeights(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = float("inf"); self.best_epoch = 0; self.weights = None

    def on_epoch_end(self, epoch, logs=None):
        vl = (logs or {}).get("val_loss")
        if vl is not None and vl < self.best:
            self.best, self.best_epoch = float(vl), epoch + 1
            self.weights = [w.copy() for w in self.model.get_weights()]


_DIA_Xtr_k = np.transpose(DIA_Xtr, (0, 2, 1))     # (N, 1, 8) -> (N, 8, 1) theo bố cục Keras
_DIA_Xva_k = np.transpose(DIA_Xva, (0, 2, 1))
_DIA_Xte_k = np.transpose(DIA_Xte, (0, 2, 1))


def dia_train_tf(seed, verbose=False):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(8, 1)),
        layers.Conv1D(16, 3, padding="same", activation="relu"),
        layers.Conv1D(16, 3, padding="same", activation="relu"),
        layers.MaxPooling1D(2), layers.Flatten(),
        layers.Dense(8, activation="relu"), layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=DIA_LR),
                  loss=keras.losses.BinaryCrossentropy(from_logits=True),
                  metrics=[keras.metrics.BinaryAccuracy(name="acc", threshold=0.0)])
    cb = DiaBestWeights()
    t0 = time.time()
    model.fit(_DIA_Xtr_k, DIA_ytr, validation_data=(_DIA_Xva_k, DIA_yva),
              epochs=DIA_EPOCHS, batch_size=DIA_BATCH, shuffle=True,
              verbose=2 if verbose else 0, callbacks=[cb])
    el = time.time() - t0
    model.set_weights(cb.weights)
    logits = model.predict(_DIA_Xte_k, batch_size=4096, verbose=0).ravel()
    return (logits >= 0.0).astype(np.int64), float(el), int(cb.best_epoch)


print("Đã định nghĩa ba hàm huấn luyện cho miền diabetes.")
print(f"Siêu tham số chung: {DIA_EPOCHS} epoch, lô {DIA_BATCH}, Adam lr = {DIA_LR}")

In [ ]:
_t_dia = time.time()
run_multi_seed("diabetes", "numpy", dia_train_numpy, DIA_yte)
print()
run_multi_seed("diabetes", "pytorch", dia_train_torch, DIA_yte)
print()
run_multi_seed("diabetes", "tensorflow", dia_train_tf, DIA_yte)
print()
print(f"Tổng thời gian miền diabetes: {time.time() - _t_dia:.1f}s")

In [ ]:
bang_da_hat_giong("diabetes")

### Diễn giải bảng `diabetes`

Miền này là phép thử khắt khe nhất cho luận điểm trung tâm, vì mô hình chỉ có 1.377 tham số. Một
mạng nhỏ như vậy có bề mặt mất mát đơn giản hơn nhiều so với mạng 509.665 tham số của miền văn bản,
nên về nguyên tắc ba khung càng dễ hội tụ về cùng một nghiệm và độ lệch chuẩn theo hạt giống càng
nhỏ. Điều đó cắt theo cả hai chiều: độ lệch chuẩn nhỏ khiến một khoảng cách nhỏ giữa hai khung vẫn
có thể vượt ngưỡng nhiễu.

Đây chính là lý do cột tỉ số ở bảng trên không đủ để kết luận, và vì sao Phần B phải dùng kiểm định
McNemar. Tỉ số chỉ so sánh khoảng cách với độ phân tán; nó không tính tới việc tập kiểm thử có
14.420 mẫu, tức mỗi mẫu chỉ đóng góp 0,0069 điểm phần trăm vào accuracy. Muốn biết chênh lệch có ý
nghĩa thống kê hay không thì phải đếm số mẫu mà hai mô hình thực sự bất đồng, và đó là việc của
McNemar.

## A.3. Miền `house_price`

Kiến trúc giống hệt `diabetes` về khung xương nhưng đầu ra **tuyến tính** và hàm mất mát là **MSE**,
đúng theo CONTRACT.md Mục 4. Siêu tham số theo ba notebook gốc: 40 epoch, lô 256, Adam với learning
rate $3 \times 10^{-3}$.

Vì đây là bài toán hồi quy, chỉ số theo dõi là $R^2$ trên thang log chứ không phải accuracy:

$$R^2 = 1 - \frac{\sum_i (\hat{y}_i - y_i)^2}{\sum_i (y_i - \bar{y})^2}$$

Hệ quả quan trọng: miền này **không** tham gia Phần B và Phần C. Kiểm định McNemar cần hai dãy dự
đoán đúng hoặc sai theo từng mẫu, còn khoảng tin cậy Wilson là khoảng cho một tỉ lệ nhị thức. Một
dự đoán hồi quy không có khái niệm "đúng" hay "sai" theo mẫu, nên áp hai công cụ đó vào đây sẽ là
dùng sai phép kiểm. Báo cáo nêu rõ điều này thay vì lặng lẽ bỏ qua miền.

In [ ]:
_raw = pd.read_csv(DATA_HOUSE)
HOU_N_RAW = len(_raw)
_h = _raw.dropna(subset=["price", "house_size", "bed", "bath"]).copy()
_h = _h[(_h["price"] >= 10_000) & (_h["price"] <= 5_000_000)]
_h = _h[(_h["house_size"] >= 200) & (_h["house_size"] <= 20_000)]
_ACRE_MEDIAN = _h["acre_lot"].median()
_h["acre_lot"] = _h["acre_lot"].fillna(_ACRE_MEDIAN).clip(lower=1e-3)

_h["log_house_size"] = np.log(_h["house_size"])
_h["total_rooms"]    = _h["bed"] + _h["bath"]
_h["bed_bath_prod"]  = _h["bed"] * _h["bath"]
_h["sqft_per_room"]  = _h["house_size"] / _h["total_rooms"].replace(0, np.nan)
_h["bath_bed_ratio"] = _h["bath"] / _h["bed"].replace(0, np.nan)
_h["log_acre_lot"]   = np.log(_h["acre_lot"])
_h["log_price"]      = np.log(_h["price"])

HOU_FEATURES = ["log_house_size", "bed", "bath", "total_rooms",
                "bed_bath_prod", "sqft_per_room", "bath_bed_ratio", "log_acre_lot"]
_h = _h.dropna(subset=HOU_FEATURES + ["log_price"])
HOU_N_CLEAN = len(_h)

_Xall = _h[HOU_FEATURES].to_numpy(dtype=np.float64)
_yall = _h["log_price"].to_numpy(dtype=np.float64)
_Xtmp, _Xte, _ytmp, _yte = train_test_split(_Xall, _yall, test_size=0.20,
                                            random_state=SPLIT_SEED)
_Xtr, _Xva, _ytr, _yva = train_test_split(_Xtmp, _ytmp, test_size=0.20,
                                          random_state=SPLIT_SEED)

_sc = StandardScaler().fit(_Xtr)
CLIP_SIGMA = 5.0                       # winsorize sau chuẩn hóa, theo đúng notebook gốc
HOU_Xtr = np.clip(_sc.transform(_Xtr), -CLIP_SIGMA, CLIP_SIGMA).reshape(-1, 1, 8).astype(np.float32)
HOU_Xva = np.clip(_sc.transform(_Xva), -CLIP_SIGMA, CLIP_SIGMA).reshape(-1, 1, 8).astype(np.float32)
HOU_Xte = np.clip(_sc.transform(_Xte), -CLIP_SIGMA, CLIP_SIGMA).reshape(-1, 1, 8).astype(np.float32)
HOU_ytr = _ytr.reshape(-1, 1).astype(np.float32)
HOU_yva = _yva.reshape(-1, 1).astype(np.float32)
HOU_yte = _yte.reshape(-1, 1).astype(np.float32)
HOU_N_TEST = len(HOU_yte)


def hou_r2(y_true_log, y_pred_log):
    """R² trên thang log, đúng định nghĩa dùng ở notebook house_price/01."""
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()
    ss_res = float(np.sum((yp - yt) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    return float(1.0 - ss_res / ss_tot)


print(f"Thô -> sạch          : {HOU_N_RAW:,} -> {HOU_N_CLEAN:,} bản ghi")
print(f"Chia tập             : train {len(HOU_ytr):,} / val {len(HOU_yva):,} "
      f"/ test {HOU_N_TEST:,}")
print(f"Trung vị acre_lot dùng để điền khuyết: {_ACRE_MEDIAN}")
print(f"Độ lệch chuẩn của log_price trên tập test: {HOU_yte.std():.6f}")
print()
print("Ghi chú: R² trên thang log của miền này chỉ vào khoảng 0,40 ở cả ba khung, vì tám đặc")
print("trưng theo hợp đồng đều thuần về cấu trúc căn nhà và không mang tín hiệu vị trí địa lý.")
print("Đó là kết quả trung thực của notebook gốc, không phải lỗi, và không được sửa ở đây.")

In [ ]:
class RegConv1D:
    """Tích chập 1 chiều 'same' cho hồi quy, khởi tạo He Normal. Sao chép từ house_price/01."""

    param_names = ["W", "b"]

    def __init__(self, c_in, c_out, k, rng):
        self.W = rng.standard_normal((c_out, c_in, k)) * np.sqrt(2.0 / (c_in * k))
        self.b = np.zeros(c_out, dtype=np.float64)
        self.k = k
        self.pad = k // 2
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self.L = X.shape[2]
        Xp = np.pad(X, ((0, 0), (0, 0), (self.pad, self.pad)))
        self.win = sliding_window_view(Xp, self.k, axis=2)
        return np.einsum("nclk,ock->nol", self.win, self.W, optimize=True) + self.b[None, :, None]

    def backward(self, dout):
        self.dW = np.einsum("nclk,nol->ock", self.win, dout, optimize=True)
        self.db = dout.sum(axis=(0, 2))
        K = self.k
        dpad = np.pad(dout, ((0, 0), (0, 0), (K - 1, K - 1)))
        wd = sliding_window_view(dpad, K, axis=2)
        W_flip = self.W[:, :, ::-1]
        dXp = np.einsum("nojk,ock->ncj", wd, W_flip, optimize=True)
        return dXp[:, :, self.pad:self.pad + self.L]


class RegReLU:
    param_names = []

    def forward(self, X):
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout):
        return dout * self.mask


class RegMaxPool1D:
    param_names = []

    def __init__(self, size=2):
        self.s = size

    def forward(self, X):
        N, C, L = X.shape
        self.in_shape = X.shape
        Lo = L // self.s
        Xr = X[:, :, :Lo * self.s].reshape(N, C, Lo, self.s)
        self.arg = Xr.argmax(axis=3)
        return Xr.max(axis=3)

    def backward(self, dout):
        N, C, L = self.in_shape
        Lo = L // self.s
        dXr = np.zeros((N, C, Lo, self.s), dtype=dout.dtype)
        n, c, l = np.ogrid[:N, :C, :Lo]
        dXr[n, c, l, self.arg] = dout
        dX = np.zeros((N, C, L), dtype=dout.dtype)
        dX[:, :, :Lo * self.s] = dXr.reshape(N, C, Lo * self.s)
        return dX


class RegFlatten:
    param_names = []

    def forward(self, X):
        self.in_shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.in_shape)


class RegDense:
    param_names = ["W", "b"]

    def __init__(self, n_in, n_out, rng, he=True):
        sigma = np.sqrt(2.0 / n_in) if he else np.sqrt(1.0 / n_in)
        self.W = rng.standard_normal((n_in, n_out)) * sigma
        self.b = np.zeros(n_out, dtype=np.float64)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self.X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T


class RegMSELoss:
    def forward(self, yhat, y):
        self.diff = yhat - y
        self.N = yhat.shape[0]
        return float(np.mean(self.diff ** 2))

    def backward(self):
        return 2.0 * self.diff / self.N


class RegAdam:
    def __init__(self, layers, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.slots = []
        for layer in layers:
            for name in getattr(layer, "param_names", []):
                p = getattr(layer, name)
                self.slots.append({"layer": layer, "name": name,
                                   "m": np.zeros_like(p), "v": np.zeros_like(p)})
        self.t = 0

    def step(self):
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t
        bc2 = 1.0 - self.b2 ** self.t
        for s in self.slots:
            g = getattr(s["layer"], "d" + s["name"])
            s["m"] = self.b1 * s["m"] + (1 - self.b1) * g
            s["v"] = self.b2 * s["v"] + (1 - self.b2) * (g * g)
            p = getattr(s["layer"], s["name"])
            setattr(s["layer"], s["name"],
                    p - self.lr * (s["m"] / bc1) / (np.sqrt(s["v"] / bc2) + self.eps))


class CNN1DRegressor:
    """CNN 1 chiều cho hồi quy, đầu ra tuyến tính. Sao chép từ house_price/01."""

    def __init__(self, seed=42):
        rng = np.random.default_rng(seed)
        self.conv1 = RegConv1D(1, 16, 3, rng)
        self.act1  = RegReLU()
        self.conv2 = RegConv1D(16, 16, 3, rng)
        self.act2  = RegReLU()
        self.pool  = RegMaxPool1D(2)
        self.flat  = RegFlatten()
        self.fc1   = RegDense(16 * 4, 8, rng, he=True)
        self.act3  = RegReLU()
        self.fc2   = RegDense(8, 1, rng, he=False)
        self.layers = [self.conv1, self.act1, self.conv2, self.act2, self.pool,
                       self.flat, self.fc1, self.act3, self.fc2]

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def n_params(self):
        return sum(getattr(l, n).size for l in self.layers
                   for n in getattr(l, "param_names", []))


HOU_N_PARAMS = CNN1DRegressor().n_params()
print(f"Số tham số CNN1DRegressor: {HOU_N_PARAMS:,} (đối chiếu hợp đồng: 1.377)")
assert HOU_N_PARAMS == 1377, "So tham so khong khop"
print("Ngăn xếp NumPy của house_price/01 đã được sao chép nguyên vẹn với tiền tố Reg.")

In [ ]:
HOU_EPOCHS, HOU_BATCH, HOU_LR = 40, 256, 3e-3

_HOU_Xtr64, _HOU_ytr64 = HOU_Xtr.astype(np.float64), HOU_ytr.astype(np.float64)


def _hou_predict_np(model, X, batch=8192):
    return np.vstack([model.forward(X[i:i + batch].astype(np.float64))
                      for i in range(0, len(X), batch)])


def hou_train_numpy(seed, verbose=False):
    model = CNN1DRegressor(seed=seed)
    crit = RegMSELoss()
    opt = RegAdam(model.layers, lr=HOU_LR)
    rng = np.random.default_rng(seed)
    best_val, best_ep, best_state = np.inf, 0, None
    t0 = time.time()
    for ep in range(1, HOU_EPOCHS + 1):
        perm = rng.permutation(len(_HOU_Xtr64))
        for i in range(0, len(perm), HOU_BATCH):
            b = perm[i:i + HOU_BATCH]
            pred = model.forward(_HOU_Xtr64[b])
            crit.forward(pred, _HOU_ytr64[b])
            model.backward(crit.backward())
            opt.step()
        va = float(np.mean((_hou_predict_np(model, HOU_Xva) - HOU_yva.astype(np.float64)) ** 2))
        if va < best_val:
            best_val, best_ep = va, ep
            best_state = [getattr(l, n).copy() for l in model.layers
                          for n in getattr(l, "param_names", [])]
        if verbose:
            print(f"    epoch {ep:2d}/{HOU_EPOCHS} val_mse={va:.6f}")
    _it = iter(best_state)
    for l in model.layers:
        for n in getattr(l, "param_names", []):
            setattr(l, n, next(_it))
    pred = _hou_predict_np(model, HOU_Xte).ravel()
    return pred, float(time.time() - t0), int(best_ep)


class CNN1DRegressorTorch(nn.Module):
    """Bản PyTorch, sao chép từ house_price/02."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 16, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool1d(kernel_size=2)
        self.flat  = nn.Flatten()
        self.fc1   = nn.Linear(16 * 4, 8)
        self.fc2   = nn.Linear(8, 1)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flat(self.pool(x))
        return self.fc2(self.relu(self.fc1(x)))


_HOU_Xtr_t = torch.from_numpy(HOU_Xtr).float().to(DEVICE)
_HOU_ytr_t = torch.from_numpy(HOU_ytr).float().to(DEVICE)
_HOU_Xva_t = torch.from_numpy(HOU_Xva).float().to(DEVICE)
_HOU_yva_t = torch.from_numpy(HOU_yva).float().to(DEVICE)
_HOU_Xte_t = torch.from_numpy(HOU_Xte).float().to(DEVICE)


def hou_train_torch(seed, verbose=False):
    torch.manual_seed(seed)
    model = CNN1DRegressorTorch().to(DEVICE)
    crit = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=HOU_LR)
    g = torch.Generator(); g.manual_seed(seed)
    best_val, best_ep, best_state = float("inf"), 0, None
    torch.cuda.synchronize(); t0 = time.time()
    for ep in range(1, HOU_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(_HOU_Xtr_t), generator=g).to(DEVICE)
        for i in range(0, len(order), HOU_BATCH):
            b = order[i:i + HOU_BATCH]
            opt.zero_grad()
            crit(model(_HOU_Xtr_t[b]), _HOU_ytr_t[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            va = float(crit(model(_HOU_Xva_t), _HOU_yva_t))
        if va < best_val:
            best_val, best_ep = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"    epoch {ep:2d}/{HOU_EPOCHS} val_mse={va:.6f}")
    torch.cuda.synchronize(); el = time.time() - t0
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pred = model(_HOU_Xte_t).cpu().numpy().ravel()
    return pred, float(el), int(best_ep)


class HouBestWeights(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = float("inf"); self.best_epoch = 0; self.weights = None

    def on_epoch_end(self, epoch, logs=None):
        vl = (logs or {}).get("val_loss")
        if vl is not None and vl < self.best:
            self.best, self.best_epoch = float(vl), epoch + 1
            self.weights = [w.copy() for w in self.model.get_weights()]


_HOU_Xtr_k = np.transpose(HOU_Xtr, (0, 2, 1))
_HOU_Xva_k = np.transpose(HOU_Xva, (0, 2, 1))
_HOU_Xte_k = np.transpose(HOU_Xte, (0, 2, 1))


def hou_train_tf(seed, verbose=False):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        layers.Input(shape=(8, 1)),
        layers.Conv1D(16, kernel_size=3, padding="same", activation=None),
        layers.ReLU(),
        layers.Conv1D(16, kernel_size=3, padding="same", activation=None),
        layers.ReLU(),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(8, activation=None),
        layers.ReLU(),
        layers.Dense(1, activation=None),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=HOU_LR),
                  loss=keras.losses.MeanSquaredError(),
                  metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
    cb = HouBestWeights()
    t0 = time.time()
    model.fit(_HOU_Xtr_k, HOU_ytr, validation_data=(_HOU_Xva_k, HOU_yva),
              epochs=HOU_EPOCHS, batch_size=HOU_BATCH, shuffle=True,
              verbose=2 if verbose else 0, callbacks=[cb])
    el = time.time() - t0
    model.set_weights(cb.weights)
    pred = model.predict(_HOU_Xte_k, batch_size=8192, verbose=0).ravel()
    return pred, float(el), int(cb.best_epoch)


def run_multi_seed_reg(domain, framework, fn):
    """Biến thể hồi quy của run_multi_seed: chỉ số là R² trên thang log."""
    seeds = SEEDS_BY_DOMAIN[domain]
    vals, tms = [], []
    print(f"[{domain} | {framework}] bắt đầu {len(seeds)} lần chạy, hạt giống {seeds}")
    for sd in seeds:
        pred, el, best_ep = chay_co_thu_lai(fn, sd)
        r2 = hou_r2(HOU_yte, pred)
        vals.append(r2); tms.append(el)
        print(f"  seed {sd} | r2 = {r2:.6f} | epoch tốt nhất = {best_ep:2d} | {el:7.1f}s")
    print(f"  => trung bình {np.mean(vals):.6f} ± {np.std(vals, ddof=1):.6f}  "
          f"(biên độ {max(vals) - min(vals):.6f}, tổng {sum(tms):.1f}s)")
    _register(domain, framework, vals, None, tms)
    return vals


print("Đã định nghĩa ba hàm huấn luyện cho miền house_price và hàm run_multi_seed_reg.")
print(f"Siêu tham số chung: {HOU_EPOCHS} epoch, lô {HOU_BATCH}, Adam lr = {HOU_LR}")

In [ ]:
_t_hou = time.time()
run_multi_seed_reg("house_price", "numpy", hou_train_numpy)
print()
run_multi_seed_reg("house_price", "pytorch", hou_train_torch)
print()
run_multi_seed_reg("house_price", "tensorflow", hou_train_tf)
print()
print(f"Tổng thời gian miền house_price: {time.time() - _t_hou:.1f}s")

In [ ]:
bang_da_hat_giong("house_price", metric="R² trên thang log")

### Diễn giải bảng `house_price`

Miền hồi quy cung cấp một góc nhìn mà ba miền phân loại không có, vì $R^2$ là đại lượng liên tục
chứ không bị lượng tử hóa theo số mẫu phân loại đúng. Ở một tập kiểm thử 30.000 mẫu, accuracy chỉ
nhận các giá trị cách nhau $1/30000$, còn $R^2$ thì thay đổi trơn. Nhờ vậy độ lệch chuẩn theo hạt
giống ở đây phản ánh đúng mức bất định của quá trình tối ưu, không bị làm mịn bởi ngưỡng quyết định.

Cần nhắc lại rằng mức $R^2$ tuyệt đối khoảng 0,40 là thấp, và nguyên nhân đã được ghi trong notebook
gốc: tám đặc trưng theo hợp đồng đều mô tả cấu trúc căn nhà, trong khi giá bất động sản phần lớn do
vị trí quyết định. Notebook này không thêm đặc trưng địa lý để kéo con số lên, vì mục tiêu ở đây là
đo **độ ổn định giữa các khung** chứ không phải tối đa hóa chất lượng mô hình. Ba khung cùng thiếu
đúng một loại thông tin, nên phép so sánh giữa chúng vẫn công bằng.

## A.4. Miền `mnist`, chỉ hai mô hình khung

Kiến trúc theo CONTRACT.md Mục 4, biến thể MNIST hai khối `32 → 64`:
`Conv-BN-ReLU-MaxPool-Dropout ×2 → Flatten → Dense(128) → ReLU → Dropout → Dense(10)`,
tổng 421.738 tham số. Siêu tham số: 10 epoch, lô 128, Adam với learning rate $10^{-3}$.

Lớp PyTorch được **nạp trực tiếp** từ `mnist/models/mnist_cnn_def.py`, là đúng tệp mà notebook
`mnist/02` đã ghi ra và notebook `mlp_vs_cnn` đang dùng lại. Cách này bảo đảm kiến trúc trùng khít
tới từng dòng thay vì trùng khít theo trí nhớ của người chép lại.

Mô hình có Dropout và BatchNorm, hai thành phần mà ba miền trước không có. Dropout là nguồn ngẫu
nhiên **thứ ba** bên cạnh khởi tạo và xáo trộn lô, nên dự đoán trước rằng độ lệch chuẩn theo hạt
giống ở miền này sẽ lớn hơn ở `diabetes`, dù accuracy tuyệt đối cao hơn nhiều.

In [ ]:
sys.path.insert(0, os.path.abspath(MNIST_MODELS))
from mnist_cnn_def import MnistCNN          # noqa: E402  (nạp lại đúng lớp của mnist/02)

_d = np.load(DATA_MNIST)
_xtr_raw, _ytr_raw = _d["x_train"], _d["y_train"].astype(np.int64)
_xte_raw, _yte_raw = _d["x_test"], _d["y_test"].astype(np.int64)

_idx_tr, _idx_va = train_test_split(np.arange(len(_xtr_raw)), test_size=0.2,
                                    stratify=_ytr_raw, random_state=SPLIT_SEED)
MNI_MEAN = float((_xtr_raw[_idx_tr].astype(np.float32) / 255.0).mean())
MNI_STD  = float((_xtr_raw[_idx_tr].astype(np.float32) / 255.0).std())


def mni_preprocess(x_u8):
    x = x_u8.astype(np.float32) / 255.0
    return ((x - MNI_MEAN) / MNI_STD)[:, None, :, :]        # bố cục NCHW


MNI_Xtr, MNI_ytr = mni_preprocess(_xtr_raw[_idx_tr]), _ytr_raw[_idx_tr]
MNI_Xva, MNI_yva = mni_preprocess(_xtr_raw[_idx_va]), _ytr_raw[_idx_va]
MNI_Xte, MNI_yte = mni_preprocess(_xte_raw), _yte_raw
MNI_N_TEST = len(MNI_yte)

del _d, _xtr_raw, _xte_raw          # ảnh thô uint8 không còn cần sau khi chuẩn hóa
gc.collect()

_preproc_ref = json.load(open(os.path.join(MNIST_MODELS, "mnist_preproc.json"),
                              encoding="utf-8"))
print(f"MEAN = {MNI_MEAN:.10f}   STD = {MNI_STD:.10f}")
print(f"Đối chiếu với mnist/models/mnist_preproc.json: mean = {_preproc_ref['mean']:.10f}, "
      f"std = {_preproc_ref['std']:.10f}")
print("Hai hằng số chuẩn hóa trùng khớp:",
      abs(MNI_MEAN - _preproc_ref["mean"]) < 1e-12 and abs(MNI_STD - _preproc_ref["std"]) < 1e-12)
print()
print(f"Train      : {MNI_Xtr.shape}  phân phối lớp {np.bincount(MNI_ytr, minlength=10).tolist()}")
print(f"Validation : {MNI_Xva.shape}")
print(f"Test       : {MNI_Xte.shape}  phân phối lớp {np.bincount(MNI_yte, minlength=10).tolist()}")
print(f"Số tham số MnistCNN: {sum(p.numel() for p in MnistCNN().parameters()):,} "
      f"(đối chiếu hợp đồng: 421.738)")

In [ ]:
MNI_EPOCHS, MNI_BATCH, MNI_LR = 10, 128, 1e-3

_MNI_Xtr_t = torch.from_numpy(MNI_Xtr).to(DEVICE)
_MNI_ytr_t = torch.from_numpy(MNI_ytr).to(DEVICE)
_MNI_Xva_t = torch.from_numpy(MNI_Xva).to(DEVICE)
_MNI_yva_t = torch.from_numpy(MNI_yva).to(DEVICE)
_MNI_Xte_t = torch.from_numpy(MNI_Xte).to(DEVICE)


def mni_train_torch(seed, verbose=False):
    """PyTorch trên GPU. Chọn epoch tốt nhất theo val_accuracy, giống notebook mnist/02."""
    torch.manual_seed(seed)
    model = MnistCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=MNI_LR)
    crit = nn.CrossEntropyLoss()
    g = torch.Generator(); g.manual_seed(seed)

    @torch.no_grad()
    def acc_on(X, y):
        model.eval()
        correct = 0
        for i in range(0, len(X), 512):
            correct += int((model(X[i:i + 512]).argmax(1) == y[i:i + 512]).sum())
        return correct / len(X)

    best_acc, best_ep, best_state = -1.0, 0, None
    torch.cuda.synchronize(); t0 = time.time()
    for ep in range(1, MNI_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(_MNI_Xtr_t), generator=g).to(DEVICE)
        for i in range(0, len(order), MNI_BATCH):
            b = order[i:i + MNI_BATCH]
            opt.zero_grad()
            crit(model(_MNI_Xtr_t[b]), _MNI_ytr_t[b]).backward()
            opt.step()
        va = acc_on(_MNI_Xva_t, _MNI_yva_t)
        if va > best_acc:
            best_acc, best_ep = va, ep
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"    epoch {ep:2d}/{MNI_EPOCHS} val_acc={va:.4f}")
    torch.cuda.synchronize(); el = time.time() - t0

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pred = torch.cat([model(_MNI_Xte_t[i:i + 512]).argmax(1)
                          for i in range(0, len(_MNI_Xte_t), 512)]).cpu().numpy()
    return pred.astype(np.int64), float(el), int(best_ep)


class MniBestWeights(keras.callbacks.Callback):
    """Chọn theo val_accuracy để khớp tiêu chí của bản PyTorch."""

    def __init__(self):
        super().__init__()
        self.best = -1.0; self.best_epoch = 0; self.weights = None

    def on_epoch_end(self, epoch, logs=None):
        va = float((logs or {}).get("val_accuracy", -1.0))
        if va > self.best:
            self.best, self.best_epoch = va, epoch + 1
            self.weights = [w.copy() for w in self.model.get_weights()]


_MNI_Xtr_k = np.transpose(MNI_Xtr, (0, 2, 3, 1))          # NCHW -> NHWC
_MNI_Xva_k = np.transpose(MNI_Xva, (0, 2, 3, 1))
_MNI_Xte_k = np.transpose(MNI_Xte, (0, 2, 3, 1))


def mni_train_tf(seed, verbose=False):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2), layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2), layers.Dropout(0.25),
        layers.Flatten(), layers.Dense(128), layers.Activation("relu"),
        layers.Dropout(0.5), layers.Dense(10),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=MNI_LR),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
    cb = MniBestWeights()
    t0 = time.time()
    model.fit(_MNI_Xtr_k, MNI_ytr, validation_data=(_MNI_Xva_k, MNI_yva),
              epochs=MNI_EPOCHS, batch_size=MNI_BATCH,
              verbose=2 if verbose else 0, callbacks=[cb])
    el = time.time() - t0
    model.set_weights(cb.weights)
    logits = model.predict(_MNI_Xte_k, batch_size=512, verbose=0)
    return logits.argmax(1).astype(np.int64), float(el), int(cb.best_epoch)


print("Đã định nghĩa hai hàm huấn luyện cho miền mnist.")
print(f"Siêu tham số chung: {MNI_EPOCHS} epoch, lô {MNI_BATCH}, Adam lr = {MNI_LR}")
print("Hai mô hình 2D thuần NumPy được bỏ qua có chủ ý, lý do đã nêu ở bảng thiết kế thực nghiệm.")

In [ ]:
_t_mni = time.time()
run_multi_seed("mnist", "pytorch", mni_train_torch, MNI_yte)
print()
run_multi_seed("mnist", "tensorflow", mni_train_tf, MNI_yte)
print()
print(f"Tổng thời gian miền mnist: {time.time() - _t_mni:.1f}s")

In [ ]:
bang_da_hat_giong("mnist")

### Diễn giải bảng `mnist`

Bảng này cần được đọc cùng một cảnh báo đã nêu ở phần thiết kế: notebook `mnist/02` chạy PyTorch
trên CPU còn notebook này chạy trên GPU theo đúng CONTRACT.md Mục 1, nên con số của hạt giống 42
không buộc phải trùng với 0,9898 đã công bố. Điều đáng quan tâm không phải là nó trùng hay không,
mà là **độ lệch chuẩn qua năm hạt giống có nuốt trọn khoảng cách 0,04 điểm phần trăm giữa PyTorch
và TensorFlow trong bảng gốc hay không**.

Ở mức accuracy quanh 99%, mỗi 0,01 điểm phần trăm tương ứng đúng **một** ảnh trong 10.000 ảnh test.
Khoảng cách 0,04 điểm phần trăm giữa hai khung ở bảng gốc vì thế chỉ là bốn ảnh. Bất kỳ độ lệch
chuẩn nào lớn hơn bốn ảnh đều khiến việc xếp hạng hai khung theo accuracy trở nên vô nghĩa, và
bảng trên cho biết chính xác con số đó.

## A.5. Tổng hợp bốn miền và hình `fig_seed_variance.png`

In [ ]:
DOMAIN_ORDER = ["customer_comments", "diabetes", "house_price", "mnist"]
DOMAIN_LABEL = {"customer_comments": "customer_comments\n(accuracy)",
                "diabetes": "diabetes\n(accuracy)",
                "house_price": "house_price\n(R² thang log)",
                "mnist": "mnist\n(accuracy)"}
FW_LABEL = {"numpy": "NumPy", "pytorch": "PyTorch", "tensorflow": "TensorFlow"}
FW_COLOR = {"numpy": "#4C72B0", "pytorch": "#DD8452", "tensorflow": "#55A868"}

fig, axes = plt.subplots(1, 4, figsize=(19, 5.6))
for ax, dom in zip(axes, DOMAIN_ORDER):
    fws = list(ACC[dom].keys())
    means = [float(np.mean(ACC[dom][f])) for f in fws]
    stds = [float(np.std(ACC[dom][f], ddof=1)) for f in fws]
    xs = np.arange(len(fws))
    ax.bar(xs, means, yerr=stds, capsize=8, width=0.58,
           color=[FW_COLOR[f] for f in fws], edgecolor="black", linewidth=0.9,
           error_kw={"elinewidth": 1.8, "ecolor": "#333333"})
    for i, f in enumerate(fws):
        ax.scatter(np.full(len(ACC[dom][f]), xs[i]), ACC[dom][f], s=26, zorder=5,
                   color="white", edgecolor="black", linewidth=0.9)
    lo = min(min(ACC[dom][f]) for f in fws)
    hi = max(max(ACC[dom][f]) for f in fws)
    pad = max((hi - lo) * 1.35, 1e-4)
    ax.set_ylim(lo - pad, hi + pad * 1.25)
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(xs[i], hi + pad * 0.45, f"{m:.4f}\n± {s:.4f}",
                ha="center", va="bottom", fontsize=9)
    ax.set_xticks(xs)
    ax.set_xticklabels([FW_LABEL[f] for f in fws], fontsize=10)
    ax.set_title(DOMAIN_LABEL[dom], fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.set_xlabel(f"{len(SEEDS_BY_DOMAIN[dom])} hạt giống: "
                  f"{SEEDS_BY_DOMAIN[dom]}", fontsize=9)
axes[0].set_ylabel("Giá trị chỉ số trên tập kiểm thử", fontsize=11)
fig.suptitle("Phương sai theo hạt giống: trung bình ± độ lệch chuẩn của từng khung\n"
             "(chấm trắng là các lần chạy riêng lẻ; trục tung được phóng to "
             "quanh vùng giá trị)", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(f"{FIG_DIR}/fig_seed_variance.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_seed_variance.png")

In [ ]:
display(Image(filename=f"{FIG_DIR}/fig_seed_variance.png"))

In [ ]:
print(f"{'Miền':<20}{'Chỉ số':<14}{'std lớn nhất':>16}{'khoảng cách khung lớn nhất':>30}"
      f"{'tỉ số':>10}")
print("-" * 90)
SUMMARY_A = {}
for dom in DOMAIN_ORDER:
    metric = "r2" if dom == "house_price" else "accuracy"
    fws = list(ACC[dom].keys())
    max_std = max(float(np.std(ACC[dom][f], ddof=1)) for f in fws)
    means = [float(np.mean(ACC[dom][f])) for f in fws]
    max_gap = float(max(means) - min(means))
    ratio = max_gap / max_std if max_std > 0 else float("nan")
    SUMMARY_A[dom] = {"max_std": max_std, "max_gap": max_gap, "ratio": ratio}
    print(f"{dom:<20}{metric:<14}{max_std:>16.6f}{max_gap:>30.6f}{ratio:>10.2f}")
print("-" * 90)
print()
_n_absorbed = sum(1 for d in SUMMARY_A.values() if d["ratio"] <= 1.0)
print(f"Số miền mà khoảng cách lớn nhất giữa hai khung KHÔNG vượt quá một độ lệch chuẩn: "
      f"{_n_absorbed}/{len(DOMAIN_ORDER)}")
print()
print("Cách đọc: cột cuối là 'khoảng cách giữa hai khung xa nhau nhất' chia cho 'độ lệch chuẩn")
print("lớn nhất do đổi hạt giống'. Tỉ số dưới 1 nghĩa là chỉ cần đổi hạt giống cũng đã tạo ra")
print("được biến động lớn hơn toàn bộ khoảng cách đang bị quy cho khung thư viện.")

### Diễn giải hình `fig_seed_variance.png` và bảng tổng hợp

Hình gồm bốn bảng con, mỗi bảng một miền, cột là khung thư viện, thanh lỗi là một độ lệch chuẩn
theo năm hạt giống. Năm chấm trắng trên mỗi cột là năm lần chạy riêng lẻ, được vẽ ra chủ ý để người
đọc thấy phân bố thật chứ không chỉ thấy một thanh lỗi đối xứng đã được làm mượt. Trục tung của mỗi
bảng con được phóng to quanh vùng giá trị, vì nếu để trục chạy từ 0 đến 1 thì cả bốn miền sẽ hiện
ra như bốn nhóm cột cao bằng nhau và toàn bộ thông tin về phương sai biến mất.

Bảng tổng hợp phía dưới ép hai đại lượng cạnh nhau: độ lệch chuẩn lớn nhất do hạt giống, và khoảng
cách lớn nhất giữa hai khung. Cột tỉ số là kết luận cô đọng nhất của Phần A. Mỗi miền có tỉ số nhỏ
hơn hoặc bằng 1 là một miền mà thứ hạng giữa các khung có thể bị đảo chỉ bằng cách đổi hạt giống,
tức là thứ hạng đó không mang thông tin.

Tuy vậy, Phần A vẫn chưa đủ để kết luận. Tỉ số này là một phép so sánh mô tả, không phải một phép
kiểm định: nó không sinh ra p-value, không có giả thuyết không, và không tận dụng được một dữ kiện
rất mạnh là cả ba khung cùng được chấm trên **đúng những mẫu giống nhau**. Phần B khai thác đúng
dữ kiện đó.

# PHẦN B. Kiểm định McNemar cho từng cặp khung

## B.1. Vì sao phải là McNemar chứ không phải kiểm định t hay kiểm định hai tỉ lệ

Ba mô hình trong mỗi miền được đánh giá trên **đúng cùng một tập kiểm thử, đúng cùng những mẫu,
theo đúng thứ tự**. Hai dãy dự đoán vì thế tương quan rất mạnh theo cấu trúc: một ảnh mờ hoặc một
câu bình luận mơ hồ sẽ làm khó cả ba mô hình cùng lúc.

Dùng kiểm định t hai mẫu hoặc kiểm định hai tỉ lệ độc lập ở đây là **sai phép kiểm**, vì cả hai đều
giả định hai mẫu độc lập. Vi phạm giả định đó làm sai số chuẩn bị ước lượng phồng lên, kiểm định
mất độ mạnh, và kết quả "không khác biệt" thu được sẽ không đáng tin vì nó có thể chỉ phản ánh một
phép kiểm sai chứ không phản ánh dữ liệu.

McNemar giải quyết đúng chỗ đó bằng cách **loại bỏ toàn bộ những mẫu mà hai mô hình đồng thuận** và
chỉ nhìn vào phần bất đồng. Lập bảng liệt kê 2×2 sau, trong đó mỗi ô đếm số mẫu của tập kiểm thử:

| | B đúng | B sai |
|---|---|---|
| **A đúng** | $n_{11}$ | $c$ |
| **A sai** | $b$ | $n_{00}$ |

Hai ô $n_{11}$ và $n_{00}$ là phần đồng thuận và bị bỏ qua hoàn toàn. Giả thuyết không là hai mô
hình sai lệch như nhau, tức $\mathbb{E}[b] = \mathbb{E}[c]$. Điều kiện trên tổng $b + c$, số đếm
$b$ tuân theo phân phối nhị thức:

$$b \mid (b+c) \;\sim\; \mathrm{Binomial}\!\left(b+c,\ \tfrac{1}{2}\right) \quad \text{dưới } H_0$$

Từ đó có hai cách tính p-value:

- **Bản chính xác (exact):** lấy trực tiếp xác suất đuôi hai phía của phân phối nhị thức,
  `scipy.stats.binomtest(b, b + c, 0.5)`. Đúng với mọi cỡ mẫu, kể cả khi $b + c$ nhỏ.
- **Bản xấp xỉ khi bình phương (chi2)** với hiệu chỉnh liên tục Yates:
  $\chi^2 = \dfrac{(|b - c| - 1)^2}{b + c}$, so với phân phối $\chi^2$ một bậc tự do. Xấp xỉ này
  chỉ đáng tin khi $b + c$ đủ lớn, thông thường là từ 25 trở lên.

**Bản được dùng trong notebook này và lý do.** Ô kết xuất ở phần cấu hình đã in ra rằng môi trường
`.venv` của dự án **không có** `statsmodels`. Theo đúng phương án dự phòng mà CONTRACT.md Mục 8.1
quy định, báo cáo dùng **bản chính xác** cài bằng `scipy.stats.binomtest(b, b + c, 0.5)`. Đây cũng
là lựa chọn an toàn hơn về mặt phương pháp, vì bản chính xác không cần giả định cỡ mẫu nào cả,
trong khi $b + c$ ở các miền tại đây chỉ ở mức vài chục tới vài trăm chứ không phải hàng nghìn. Giá
phải trả là bản chính xác thận trọng hơn một chút, tức p-value hơi cao hơn bản khi bình phương, nên
nếu nó vẫn kết luận "có khác biệt" thì kết luận đó càng chắc. Bảng kết quả vẫn in kèm giá trị
$\chi^2$ để đối chiếu, nhưng con số ghi vào tệp JSON là của bản chính xác.

In [ ]:
def mcnemar_pair(y_true, pred_a, pred_b, alpha=ALPHA):
    """Bảng bất đồng 2x2 và kiểm định McNemar cho hai bộ phân loại trên CÙNG tập kiểm thử.

    b = số mẫu A sai và B đúng ; c = số mẫu A đúng và B sai.
    Trả về cả bản chính xác (nhị thức) lẫn giá trị chi2 có hiệu chỉnh Yates để đối chiếu.
    """
    ok_a = np.asarray(pred_a) == np.asarray(y_true)
    ok_b = np.asarray(pred_b) == np.asarray(y_true)
    n11 = int(np.sum(ok_a & ok_b))
    n00 = int(np.sum(~ok_a & ~ok_b))
    b = int(np.sum(~ok_a & ok_b))
    c = int(np.sum(ok_a & ~ok_b))
    n_disc = b + c

    if n_disc == 0:
        p_exact, chi2_stat, p_chi2 = 1.0, 0.0, 1.0
    else:
        if HAS_STATSMODELS:
            p_exact = float(sm_mcnemar([[n11, c], [b, n00]], exact=True).pvalue)
        else:
            p_exact = float(stats.binomtest(b, n_disc, 0.5).pvalue)
        chi2_stat = float((abs(b - c) - 1) ** 2 / n_disc) if n_disc > 0 else 0.0
        p_chi2 = float(stats.chi2.sf(chi2_stat, df=1))

    return {"n11": n11, "n00": n00, "b": b, "c": c, "n_discordant": n_disc,
            "statistic": float(min(b, c)), "p_value": p_exact,
            "significant": bool(p_exact < alpha),
            "chi2": chi2_stat, "p_chi2": p_chi2,
            "method": "exact", "backend": MCNEMAR_BACKEND}


# Kiểm tra tính đúng đắn của hàm trên hai trường hợp có đáp số biết trước
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
_same = mcnemar_pair(_y, _y.copy(), _y.copy())
print("Trường hợp 1, hai mô hình dự đoán y hệt nhau:")
print(f"  b = {_same['b']}, c = {_same['c']}, p = {_same['p_value']:.6f}, "
      f"kết luận khác biệt: {_same['significant']}")
assert _same["p_value"] == 1.0 and not _same["significant"]

_pa = _y.copy(); _pb = _y.copy()
_pb[:6] = 1 - _pb[:6]        # B sai 6 mẫu mà A đúng  -> c = 6, b = 0
_case2 = mcnemar_pair(_y, _pa, _pb)
_p_hand = float(stats.binomtest(0, 6, 0.5).pvalue)
print()
print("Trường hợp 2, B sai đúng 6 mẫu mà A làm đúng, A không sai mẫu nào B đúng:")
print(f"  b = {_case2['b']}, c = {_case2['c']}, p tính bằng hàm = {_case2['p_value']:.8f}")
print(f"  p tính tay bằng binomtest(0, 6, 0.5)  = {_p_hand:.8f}  -> "
      f"2 x 0,5^6 = {2 * 0.5 ** 6:.8f}")
assert abs(_case2["p_value"] - _p_hand) < 1e-12
print()
print("Hàm mcnemar_pair khớp với đáp số tính tay trên cả hai trường hợp đối chứng.")

In [ ]:
CLS_DOMAINS = ["customer_comments", "diabetes", "mnist"]
Y_TRUE = {"customer_comments": CMT_yte.astype(np.int64),
          "diabetes": DIA_yte.astype(np.int64),
          "mnist": MNI_yte.astype(np.int64)}
REF_SEED = SEEDS[0]        # hạt giống 42, đúng cấu hình mà các bảng đối chuẩn đã công bố

MCNEMAR = {}
print(f"Kiểm định McNemar trên dự đoán của hạt giống {REF_SEED}, "
      f"mức ý nghĩa alpha = {ALPHA}")
print(f"Bản kiểm định: CHÍNH XÁC (nhị thức) qua {MCNEMAR_BACKEND}")
print("=" * 112)
for dom in CLS_DOMAINS:
    fws = list(PRED[dom].keys())
    n = len(Y_TRUE[dom])
    print(f"\n### Miền {dom}  (n_test = {n:,})")
    print(f"{'Cặp khung':<26}{'n11':>8}{'n00':>7}{'b':>6}{'c':>6}{'b+c':>7}"
          f"{'thống kê':>11}{'p (exact)':>13}{'chi2':>9}{'p (chi2)':>12}{'Kết luận':>16}")
    print("-" * 112)
    rows = []
    for i in range(len(fws)):
        for j in range(i + 1, len(fws)):
            a, bb = fws[i], fws[j]
            r = mcnemar_pair(Y_TRUE[dom], PRED[dom][a][REF_SEED], PRED[dom][bb][REF_SEED])
            r["pair"] = [a, bb]
            # Độ bền của kết luận qua mọi hạt giống của miền, ghép cặp theo từng hạt giống
            dom_seeds = SEEDS_BY_DOMAIN[dom]
            n_sig_seeds = sum(
                1 for sd in dom_seeds
                if mcnemar_pair(Y_TRUE[dom], PRED[dom][a][sd], PRED[dom][bb][sd])["significant"])
            r["n_seeds_significant"] = int(n_sig_seeds)
            r["n_seeds_tested"] = len(dom_seeds)
            rows.append(r)
            verdict = "KHÁC BIỆT" if r["significant"] else "không khác biệt"
            print(f"{a + ' vs ' + bb:<26}{r['n11']:>8,}{r['n00']:>7,}{r['b']:>6}{r['c']:>6}"
                  f"{r['n_discordant']:>7}{r['statistic']:>11.1f}{r['p_value']:>13.6f}"
                  f"{r['chi2']:>9.3f}{r['p_chi2']:>12.6f}{verdict:>16}")
    MCNEMAR[dom] = rows

_all = [r for rows in MCNEMAR.values() for r in rows]
N_PAIRS = len(_all)
N_SIG = sum(1 for r in _all if r["significant"])
print()
print("=" * 112)
print(f"TỔNG KẾT: {N_SIG}/{N_PAIRS} cặp khung khác biệt có ý nghĩa thống kê ở mức "
      f"alpha = {ALPHA}")
print(f"          {N_PAIRS - N_SIG}/{N_PAIRS} cặp KHÔNG bác bỏ được giả thuyết không.")

In [ ]:
print("Độ bền của kết luận khi lặp lại phép kiểm trên mọi hạt giống của miền")
print("(mỗi hạt giống cho một cặp mô hình mới, vẫn trên đúng tập kiểm thử đó)")
print()
print(f"{'Miền':<20}{'Cặp khung':<26}{'số hạt giống khác biệt':>26}{'kết luận ở seed 42':>22}")
print("-" * 94)
for dom in CLS_DOMAINS:
    for r in MCNEMAR[dom]:
        print(f"{dom:<20}{r['pair'][0] + ' vs ' + r['pair'][1]:<26}"
              f"{str(r['n_seeds_significant']) + '/' + str(r['n_seeds_tested']):>26}"
              f"{('KHÁC BIỆT' if r['significant'] else 'không khác biệt'):>22}")
print("-" * 94)
_stable = sum(1 for r in _all
              if r["n_seeds_significant"] in (0, r["n_seeds_tested"]))
print()
print(f"Số cặp cho kết luận NHẤT QUÁN trên mọi hạt giống của miền: {_stable}/{N_PAIRS}")
print("Cột giữa là bằng chứng bổ sung quan trọng: một cặp chỉ khác biệt ở một vài hạt giống thì")
print("khác biệt đó gắn với lần khởi tạo cụ thể chứ không gắn với khung thư viện.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.8))
for ax, dom in zip(axes, CLS_DOMAINS):
    fws = list(PRED[dom].keys())
    k = len(fws)
    M = np.full((k, k), np.nan)
    for r in MCNEMAR[dom]:
        i, j = fws.index(r["pair"][0]), fws.index(r["pair"][1])
        M[i, j] = M[j, i] = r["p_value"]

    im = ax.imshow(np.ma.masked_invalid(M), cmap="RdYlGn", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(k)); ax.set_xticklabels([FW_LABEL[f] for f in fws], fontsize=10)
    ax.set_yticks(range(k)); ax.set_yticklabels([FW_LABEL[f] for f in fws], fontsize=10)
    for r in MCNEMAR[dom]:
        i, j = fws.index(r["pair"][0]), fws.index(r["pair"][1])
        txt = f"p = {r['p_value']:.4f}\nb = {r['b']}, c = {r['c']}"
        if r["significant"]:
            txt += "\nKHÁC BIỆT"
        for (u, v) in ((i, j), (j, i)):
            ax.text(v, u, txt, ha="center", va="center", fontsize=9,
                    fontweight="bold" if r["significant"] else "normal")
            if r["significant"]:
                ax.add_patch(plt.Rectangle((v - 0.5, u - 0.5), 1, 1, fill=False,
                                           edgecolor="#8B0000", linewidth=4))
    for i in range(k):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, facecolor="#DDDDDD",
                                   edgecolor="white"))
        ax.text(i, i, "cùng\nmô hình", ha="center", va="center", fontsize=9, color="#555555")
    ax.set_title(f"{dom}\n(n_test = {len(Y_TRUE[dom]):,})", fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(-0.5, k, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", length=0)

cbar = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("p-value của kiểm định McNemar (bản chính xác)", fontsize=11)
cbar.ax.axhline(ALPHA, color="#8B0000", linewidth=2.5)
cbar.ax.text(1.6, ALPHA, f" alpha = {ALPHA}", color="#8B0000", fontsize=10,
             va="center", transform=cbar.ax.get_yaxis_transform())
fig.suptitle("Ma trận p-value McNemar từng cặp khung, dự đoán của hạt giống "
             f"{REF_SEED}\nÔ viền đỏ đậm là cặp khác biệt có ý nghĩa thống kê; "
             "màu càng xanh thì càng không có bằng chứng về khác biệt",
             fontsize=13, fontweight="bold")
plt.savefig(f"{FIG_DIR}/fig_mcnemar_matrix.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_mcnemar_matrix.png")
print(f"Số ô được viền đỏ (cặp khác biệt có ý nghĩa): {N_SIG}/{N_PAIRS}")

In [ ]:
display(Image(filename=f"{FIG_DIR}/fig_mcnemar_matrix.png"))

### Diễn giải hình `fig_mcnemar_matrix.png`

Ma trận đối xứng vì McNemar không phân biệt thứ tự hai mô hình: hoán đổi vai trò A và B chỉ hoán
đổi $b$ với $c$, còn p-value giữ nguyên. Đường chéo được tô xám và không mang thông tin, vì so sánh
một mô hình với chính nó luôn cho $b = c = 0$.

Mỗi ô in ba dòng: p-value, cặp số $(b, c)$, và nhãn `KHÁC BIỆT` nếu bác bỏ được giả thuyết không.
Cặp $(b, c)$ mới là phần cần đọc kỹ nhất, vì nó cho biết **quy mô thật của sự bất đồng**. Chẳng
hạn, hai mô hình chênh nhau vài chục mẫu trên một tập kiểm thử hàng chục nghìn mẫu nghĩa là chúng
đồng thuận trên hơn 99% số mẫu, và toàn bộ cuộc tranh luận về khung thư viện nào tốt hơn diễn ra
trên đúng phần bất đồng nhỏ đó.

Thang màu đi từ đỏ tại $p = 0$ tới xanh lá tại $p = 1$, với vạch đỏ trên thanh màu đánh dấu ngưỡng
$\alpha = 0{,}05$. Cần nói rõ một điểm dễ hiểu sai: một ô xanh **không chứng minh** hai mô hình
giống nhau, nó chỉ nói rằng dữ liệu hiện có không đủ bằng chứng để khẳng định chúng khác nhau. Sự
phân biệt này quan trọng, và báo cáo giữ đúng cách phát biểu đó thay vì nói quá lên thành "ba khung
tương đương nhau".

# PHẦN C. Khoảng tin cậy Wilson 95% cho mọi giá trị accuracy

## C.1. Vì sao dùng Wilson chứ không dùng Wald

Khoảng Wald là công thức quen thuộc nhất và cũng là công thức hỏng nhất ở đúng vùng mà báo cáo này
quan tâm:

$$\text{Wald:}\quad \hat{p} \pm z_{0{,}975}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

Nó có ba khuyết điểm nặng. Thứ nhất, **hai cận có thể tràn ra ngoài đoạn $[0, 1]$**, tạo ra những
phát biểu vô nghĩa kiểu "accuracy nằm trong khoảng từ 98,7% đến 100,3%". Thứ hai, **độ phủ thực tế
tụt xa dưới mức 95% danh nghĩa khi $\hat{p}$ tiến gần 0 hoặc 1**, mà đó chính là chế độ của một mô
hình MNIST đạt 99%. Thứ ba, ở trường hợp cực đoan $\hat{p} = 1$ thì $\hat{p}(1-\hat{p}) = 0$, nên
Wald thu về một điểm duy nhất và khẳng định độ chắc chắn tuyệt đối, điều hiển nhiên là sai.

Khoảng Wilson khắc phục cả ba bằng cách giải trực tiếp bất phương trình
$\left|\hat{p} - p\right| \le z\sqrt{p(1-p)/n}$ theo ẩn $p$ thay vì thay $p$ bằng $\hat{p}$ trong
sai số chuẩn:

$$
\text{Wilson:}\quad
\frac{\hat{p} + \dfrac{z^2}{2n}}{1 + \dfrac{z^2}{n}}
\;\pm\;
\frac{z}{1 + \dfrac{z^2}{n}}\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}
$$

Hệ quả là Wilson luôn nằm trọn trong $[0, 1]$, giữ được độ phủ gần 95% ngay cả khi tỉ lệ sát biên,
và vẫn cho một khoảng có bề rộng dương khi $\hat{p} = 1$. Tâm của khoảng Wilson bị kéo nhẹ về phía
0,5 so với $\hat{p}$, và đó là hành vi đúng chứ không phải sai lệch: nó phản ánh việc một tỉ lệ
quan sát sát biên thường là ước lượng lạc quan quá mức.

Ô kết xuất dưới đây tính cả hai khoảng cạnh nhau để người đọc thấy khác biệt bằng số thật của chính
các mô hình trong báo cáo này, chứ không chỉ đọc một lập luận lý thuyết.

In [ ]:
def wilson_ci(k, n, z=Z95):
    """Khoảng tin cậy Wilson cho tỉ lệ nhị thức. k là số mẫu đúng, n là cỡ mẫu."""
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1.0 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    margin = (z / denom) * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float(p), float(center - margin), float(center + margin)


def wald_ci(k, n, z=Z95):
    """Khoảng Wald, chỉ dùng để đối chiếu; hai cận có thể tràn ra ngoài [0, 1]."""
    p = k / n
    half = z * math.sqrt(p * (1 - p) / n)
    return float(p), float(p - half), float(p + half)


# Đối chứng trên trường hợp cực đoan p_hat = 1, nơi Wald sụp đổ hoàn toàn
_p, _lo, _hi = wilson_ci(100, 100)
_pw, _low, _hiw = wald_ci(100, 100)
print("Trường hợp cực đoan: 100 mẫu đúng trên 100 mẫu (p_hat = 1,0)")
print(f"  Wilson : [{_lo:.6f}, {_hi:.6f}]   bề rộng {_hi - _lo:.6f}")
print(f"  Wald   : [{_low:.6f}, {_hiw:.6f}]   bề rộng {_hiw - _low:.6f}  <-- thu về một điểm")
print()
_p, _lo, _hi = wilson_ci(9990, 10000)
_pw, _low, _hiw = wald_ci(9990, 10000)
print("Trường hợp sát biên giống mô hình MNIST: 9 990 mẫu đúng trên 10 000")
print(f"  Wilson : [{_lo:.6f}, {_hi:.6f}]")
print(f"  Wald   : [{_low:.6f}, {_hiw:.6f}]")
print(f"  Tâm Wilson lệch khỏi p_hat một lượng {abs((_lo + _hi) / 2 - _p):.6f}, "
      f"bị kéo về phía 0,5 đúng như lý thuyết mô tả.")
print()
print("Hai hàm wilson_ci và wald_ci đã được đối chứng, sẵn sàng áp lên dữ liệu thật.")

In [ ]:
WILSON = {}
print(f"Khoảng tin cậy 95% cho accuracy của hạt giống {REF_SEED}")
print("=" * 118)
print(f"{'Miền':<20}{'Khung':<13}{'n':>9}{'số đúng':>10}{'accuracy':>12}"
      f"{'Wilson lo':>12}{'Wilson hi':>12}{'bề rộng':>11}{'Wald lo':>12}{'Wald hi':>12}")
print("-" * 118)
for dom in CLS_DOMAINS:
    WILSON[dom] = {}
    y = Y_TRUE[dom]
    n = len(y)
    for fw in PRED[dom]:
        k = int(np.sum(PRED[dom][fw][REF_SEED] == y))
        p, lo, hi = wilson_ci(k, n)
        _, wlo, whi = wald_ci(k, n)
        WILSON[dom][fw] = {"accuracy": p, "n": int(n), "lo": lo, "hi": hi,
                           "n_correct": k, "wald_lo": wlo, "wald_hi": whi}
        print(f"{dom:<20}{fw:<13}{n:>9,}{k:>10,}{p:>12.6f}{lo:>12.6f}{hi:>12.6f}"
              f"{hi - lo:>11.6f}{wlo:>12.6f}{whi:>12.6f}")
print("-" * 118)

print()
print("Kiểm tra giao nhau giữa các khoảng Wilson trên cùng một miền")
print(f"{'Miền':<20}{'Cặp khung':<26}{'giao nhau':>12}{'bề rộng phần giao':>22}"
      f"{'McNemar':>18}")
print("-" * 98)
OVERLAP = []
for dom in CLS_DOMAINS:
    fws = list(WILSON[dom].keys())
    for i in range(len(fws)):
        for j in range(i + 1, len(fws)):
            a, b_ = WILSON[dom][fws[i]], WILSON[dom][fws[j]]
            inter = min(a["hi"], b_["hi"]) - max(a["lo"], b_["lo"])
            ov = inter > 0
            mc = next(r for r in MCNEMAR[dom]
                      if set(r["pair"]) == {fws[i], fws[j]})
            OVERLAP.append(ov)
            print(f"{dom:<20}{fws[i] + ' vs ' + fws[j]:<26}"
                  f"{('CÓ' if ov else 'KHÔNG'):>12}{inter:>22.6f}"
                  f"{('KHÁC BIỆT' if mc['significant'] else 'không khác biệt'):>18}")
print("-" * 98)
print(f"Số cặp có khoảng Wilson giao nhau: {sum(OVERLAP)}/{len(OVERLAP)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.4))
for ax, dom in zip(axes, CLS_DOMAINS):
    fws = list(WILSON[dom].keys())
    ys = np.arange(len(fws))[::-1]
    for yv, fw in zip(ys, fws):
        w = WILSON[dom][fw]
        # Dải nền trải hết chiều cao để phần giao nhau giữa các khoảng hiện ra thành vùng đậm
        ax.axvspan(w["lo"], w["hi"], color=FW_COLOR[fw], alpha=0.18, zorder=1)
        ax.errorbar(w["accuracy"], yv,
                    xerr=[[w["accuracy"] - w["lo"]], [w["hi"] - w["accuracy"]]],
                    fmt="o", markersize=10, capsize=10, capthick=2.4, elinewidth=2.4,
                    color=FW_COLOR[fw], zorder=4, markeredgecolor="black")
        ax.text(w["accuracy"], yv + 0.22, f"{w['accuracy']:.4f}", ha="center",
                va="bottom", fontsize=10, fontweight="bold")
        ax.text(w["lo"], yv - 0.26, f"{w['lo']:.4f}", ha="center", va="top", fontsize=8.5)
        ax.text(w["hi"], yv - 0.26, f"{w['hi']:.4f}", ha="center", va="top", fontsize=8.5)
    ax.set_yticks(ys); ax.set_yticklabels([FW_LABEL[f] for f in fws], fontsize=11)
    ax.set_ylim(-0.75, len(fws) - 0.25)
    lo = min(WILSON[dom][f]["lo"] for f in fws)
    hi = max(WILSON[dom][f]["hi"] for f in fws)
    pad = (hi - lo) * 0.22
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_xlabel("Accuracy trên tập kiểm thử", fontsize=11)
    ax.set_title(f"{dom}\n(n_test = {len(Y_TRUE[dom]):,})", fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.3, linestyle="--")
fig.suptitle("Khoảng tin cậy Wilson 95% cho accuracy, hạt giống "
             f"{REF_SEED}\nDải màu trải dọc là khoảng tin cậy; vùng có nhiều dải chồng lên nhau "
             "chính là phần giao nhau giữa các khung", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig(f"{FIG_DIR}/fig_wilson_ci.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu:", f"{FIG_DIR}/fig_wilson_ci.png")

In [ ]:
display(Image(filename=f"{FIG_DIR}/fig_wilson_ci.png"))

### Diễn giải hình `fig_wilson_ci.png`

Hình được vẽ theo cách làm cho **sự giao nhau trở thành thứ đập vào mắt trước tiên**, vì giao nhau
mới là thông điệp chứ không phải vị trí của từng điểm. Mỗi khoảng tin cậy được vẽ thành một dải màu
trải suốt chiều cao bảng con, nên chỗ nào hai hay ba dải chồng lên nhau thì màu ở đó đậm hẳn lên và
người đọc thấy ngay phần giá trị accuracy mà cả các khung đều còn tương thích với dữ liệu.

Bảng số phía trên cung cấp phần định lượng của cùng thông điệp đó, qua cột bề rộng phần giao. Cần
lưu ý một điểm kỹ thuật thường bị dùng sai: **hai khoảng tin cậy giao nhau là bằng chứng gợi ý chứ
không phải một phép kiểm**. Quy tắc "giao nhau thì không khác biệt" quá bảo thủ vì nó bỏ qua tương
quan giữa hai ước lượng, và đó đúng là lý do Phần B phải tồn tại. Hai phần bổ trợ nhau: Wilson cho
biết **độ chính xác của từng ước lượng**, còn McNemar cho biết **có bằng chứng về khác biệt giữa
hai ước lượng hay không**.

Cột `bề rộng` cũng đáng đọc riêng. Bề rộng của khoảng tỉ lệ nghịch với căn bậc hai của cỡ mẫu, nên
miền có tập kiểm thử lớn sẽ có khoảng hẹp hơn hẳn. Đây là lời nhắc rằng con số accuracy trên một
tập kiểm thử nhỏ mang ít thông tin hơn nhiều so với cùng con số đó trên một tập lớn, dù cả hai đều
được in ra với sáu chữ số thập phân như nhau.

# PHẦN D. Ghi tệp JSON và kết luận

In [ ]:
metrics = {"multi_seed": {}, "mcnemar": {}, "wilson": {}, "notes": ""}

for dom in DOMAIN_ORDER:
    metric = "r2" if dom == "house_price" else "accuracy"
    metrics["multi_seed"][dom] = {}
    for fw, vals in ACC[dom].items():
        metrics["multi_seed"][dom][fw] = {
            "metric": metric,
            "seeds": [int(s) for s in SEEDS_BY_DOMAIN[dom]],
            "values": [float(v) for v in vals],
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals, ddof=1)),
        }

for dom in CLS_DOMAINS:
    metrics["mcnemar"][dom] = [{
        "pair": [r["pair"][0], r["pair"][1]],
        "b": int(r["b"]),
        "c": int(r["c"]),
        "statistic": float(r["statistic"]),
        "p_value": float(r["p_value"]),
        "significant": bool(r["significant"]),
        "method": r["method"],
    } for r in MCNEMAR[dom]]

for dom in CLS_DOMAINS:
    metrics["wilson"][dom] = {
        fw: {"accuracy": float(w["accuracy"]), "n": int(w["n"]),
             "lo": float(w["lo"]), "hi": float(w["hi"])}
        for fw, w in WILSON[dom].items()
    }

_ratios = ", ".join(f"{d} {SUMMARY_A[d]['ratio']:.2f}" for d in DOMAIN_ORDER)
_seed_note = ", ".join(f"{d} {len(SEEDS_BY_DOMAIN[d])} hat giong {SEEDS_BY_DOMAIN[d]}"
                       for d in DOMAIN_ORDER)
metrics["notes"] = (
    f"Da hat giong theo tung mien: {_seed_note}. Hai mien customer_comments va diabetes giu du "
    f"nam hat giong vi bien do chenh lech giua cac khung o do sit sao nhat; house_price va mnist "
    f"dung ba hat giong vi may dang chay song song nhieu notebook nang va mot lan chay truoc do "
    f"da that bai voi MemoryError o cap he thong. Do lech chuan uoc luong tu ba quan sat kem chac "
    f"hon tu nam, va bao cao noi ro dieu do thay vi lam ngo. "
    f"Kien truc, sieu tham "
    f"so va phep chia train/val/test giu nguyen theo CONTRACT.md, chi hat giong khoi tao va thu tu "
    f"xao tron lo thay doi. Phep chia du lieu co dinh o random_state={SPLIT_SEED} vi kiem dinh "
    f"McNemar bat buoc hai mo hinh chay tren cung cac mau. "
    f"Ti so (khoang cach khung lon nhat)/(do lech chuan lon nhat) theo mien: {_ratios}. "
    f"McNemar: ban CHINH XAC qua {MCNEMAR_BACKEND} vi moi truong .venv khong co statsmodels, "
    f"dung phuong an du phong ma CONTRACT.md Muc 8.1 quy dinh; statistic = min(b, c); "
    f"{N_SIG}/{N_PAIRS} cap khac biet co y nghia o alpha={ALPHA}; "
    f"{_stable}/{N_PAIRS} cap cho ket luan nhat quan tren ca nam hat giong. "
    f"McNemar va Wilson deu tinh tren du doan cua hat giong {REF_SEED}, dung cau hinh ma cac bang "
    f"doi chuan o cac chuong truoc da cong bo. "
    f"house_price la bai toan hoi quy nen khong co McNemar va khong co Wilson; chi so la R2 tren "
    f"thang log. "
    f"Mien mnist chi lap lai hai mo hinh framework; hai mo hinh 2D thuan NumPy bi bo qua co chu y "
    f"vi mot lan chay da ton 120s va 148s. "
    f"SAI LECH CO GHI NHAN: notebook mnist/02 chay PyTorch tren CPU, notebook nay chay tren GPU "
    f"theo CONTRACT.md Muc 1, nen con so cua seed 42 o mien mnist khong trung khit voi 0.9898 da "
    f"cong bo; kien truc, phep chia, sieu tham so va thu tu lo deu giu nguyen. "
    f"Rieng TxtConv1D duoc them co optimize=True cho ba loi goi einsum (do duoc 3x den 7.5x nhanh "
    f"hon, sai lech tuong doi khoang 1e-14, thuan tuy tai ket hop dau phay dong); hai ngan xep "
    f"NumPy con lai von da dung co nay. "
    f"Cac lop NumPy duoc sao chep tu notebook goc va them tien to Txt/Tab/Reg de bon ngan xep cung "
    f"ton tai trong mot notebook; phan toan hoc giu nguyen. Lop MnistCNN duoc nap truc tiep tu "
    f"mnist/models/mnist_cnn_def.py. "
    f"Thoi gian chay khong the so sanh giua cac khung vi PyTorch chay GPU con TensorFlow va NumPy "
    f"chay CPU."
)

OUT_JSON = f"{REP_DIR}/metrics_statistical.json"
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Đã ghi:", OUT_JSON)
print(f"Kích thước tệp: {os.path.getsize(OUT_JSON):,} byte")
print()
print("Kiểm tra lại lược đồ bằng cách nạp ngược tệp vừa ghi:")
_chk = json.load(open(OUT_JSON, encoding="utf-8"))
print("  khóa cấp một     :", list(_chk.keys()))
print("  miền multi_seed  :", list(_chk["multi_seed"].keys()))
print("  miền mcnemar     :", list(_chk["mcnemar"].keys()))
print("  miền wilson      :", list(_chk["wilson"].keys()))
print("  số mục multi_seed:",
      sum(len(v) for v in _chk["multi_seed"].values()))
print("  số cặp mcnemar   :", sum(len(v) for v in _chk["mcnemar"].values()))
print("  số mục wilson    :", sum(len(v) for v in _chk["wilson"].values()))
print()
print("Ví dụ một mục multi_seed:")
print(json.dumps(_chk["multi_seed"]["mnist"], ensure_ascii=False, indent=2))
print()
print("Ví dụ một mục mcnemar:")
print(json.dumps(_chk["mcnemar"]["mnist"], ensure_ascii=False, indent=2))

In [ ]:
print("BẢNG TỔNG KẾT CUỐI CÙNG")
print("=" * 96)
print(f"{'Miền':<20}{'Chỉ số':<10}{'Khung':<13}{'trung bình ± độ lệch chuẩn':>32}"
      f"{'khoảng Wilson 95%':>21}")
print("-" * 96)
for dom in DOMAIN_ORDER:
    metric = "R²" if dom == "house_price" else "acc"
    for fw, vals in ACC[dom].items():
        ci = ""
        if dom in WILSON:
            w = WILSON[dom][fw]
            ci = f"[{w['lo']:.4f}, {w['hi']:.4f}]"
        print(f"{dom:<20}{metric:<10}{fw:<13}"
              f"{f'{np.mean(vals):.6f} ± {np.std(vals, ddof=1):.6f}':>32}{ci:>21}")
print("-" * 96)
print()
print(f"Số cặp khung đã kiểm định McNemar          : {N_PAIRS}")
print(f"Số cặp KHÁC BIỆT có ý nghĩa (alpha = {ALPHA}) : {N_SIG}")
print(f"Số cặp KHÔNG khác biệt có ý nghĩa          : {N_PAIRS - N_SIG}")
print(f"Tỉ lệ cặp không khác biệt                  : {(N_PAIRS - N_SIG) / N_PAIRS:.1%}")
print(f"Số cặp có khoảng Wilson giao nhau          : {sum(OVERLAP)}/{len(OVERLAP)}")
print(f"Số miền có khoảng cách khung nhỏ hơn 1 std : {_n_absorbed}/{len(DOMAIN_ORDER)}")
print("=" * 96)
print()
DEVICE_LABEL = {"numpy": "CPU", "pytorch": f"GPU ({torch.cuda.get_device_name(0)})",
                "tensorflow": "CPU"}

print("Tổng thời gian huấn luyện đã tiêu tốn cho toàn bộ thực nghiệm đa hạt giống")
print("(cột thiết bị là bắt buộc, vì ba khung KHÔNG chạy trên cùng phần cứng):")
print()
print(f"  {'Miền':<20}{'Khung':<13}{'Thiết bị':<32}{'tổng':>9}{'trung bình mỗi lần':>21}")
print("  " + "-" * 93)
_tot = 0.0
for dom in DOMAIN_ORDER:
    for fw, tms in TIMES[dom].items():
        _tot += float(np.sum(tms))
        print(f"  {dom:<20}{fw:<13}{DEVICE_LABEL[fw]:<32}{np.sum(tms):>8.1f}s"
              f"{f'{np.mean(tms):.1f}s x {len(tms)} lần':>21}")
print("  " + "-" * 93)
print(f"  {'TỔNG CỘNG':<65}{_tot:>8.1f}s  ({_tot / 60:.1f} phút)")
print()
print("Nhắc lại theo CONTRACT.md Mục 1: cột thời gian KHÔNG so sánh được giữa các khung,")
print("vì PyTorch chạy GPU còn TensorFlow và NumPy chạy CPU. Các chỉ số chất lượng thì so")
print("sánh được bình thường vì không phụ thuộc thiết bị.")
print()
print("Thêm một lưu ý đọc số: mức lợi của GPU phụ thuộc mạnh vào kích cỡ mô hình. Với hai mô")
print("hình 1.377 tham số của diabetes và house_price, chi phí phát lệnh xuống GPU chiếm phần")
print("lớn thời gian nên mức lợi chỉ ở mức khiêm tốn; với CNN 2D 421.738 tham số của mnist thì")
print("mức lợi lớn hơn hẳn. Bảng trên phản ánh đúng hiện tượng đó.")

## Kết luận

Notebook này mở đầu bằng một câu hỏi mà các bảng đối chuẩn ở những chương trước đã né tránh, rằng
chênh lệch vài phần mười điểm phần trăm giữa hai khung thư viện có phải là khác biệt thật hay
không. Ba công cụ đã được dùng để trả lời, và cả ba đều dựa trên số liệu của những lần chạy thật
được in đầy đủ ở các ô phía trên.

**Phần A** cho thấy độ lệch chuẩn của chỉ số khi chỉ đổi hạt giống nằm ở cùng bậc độ lớn với khoảng
cách giữa các khung. Bảng tổng hợp in ra tỉ số cụ thể của từng miền. Mỗi miền có tỉ số không vượt
quá 1 là một miền mà bảng xếp hạng ba khung có thể bị đảo trật tự chỉ bằng thao tác đổi một con số
hạt giống, tức là bảng xếp hạng đó không mang thông tin về khung thư viện.

**Phần B** chuyển từ so sánh mô tả sang kiểm định thống kê đúng nghĩa. Kiểm định McNemar được chọn
vì ba mô hình được chấm trên cùng những mẫu, khiến hai dãy dự đoán tương quan mạnh và làm hỏng giả
định độc lập của kiểm định t hay kiểm định hai tỉ lệ. Vì môi trường không có `statsmodels`, báo cáo
dùng bản chính xác cài bằng `scipy.stats.binomtest(b, b + c, 0.5)`, đúng phương án dự phòng mà hợp
đồng quy định, và đây cũng là bản không cần giả định cỡ mẫu. Bảng kết quả ghi rõ số cặp khác biệt
có ý nghĩa trên tổng số cặp đã kiểm, kèm bảng độ bền cho biết kết luận có giữ nguyên qua cả năm hạt
giống hay không.

**Phần C** bổ sung khoảng tin cậy Wilson cho mọi giá trị accuracy. Wilson được chọn thay Wald vì
Wald tràn ra ngoài đoạn $[0,1]$ và mất độ phủ đúng tại vùng tỉ lệ sát 1, là vùng làm việc của mô
hình MNIST 99%. Phép đối chứng số học ngay trong notebook cho thấy Wald thu khoảng về một điểm duy
nhất khi $\hat{p} = 1$, một kết quả vô nghĩa mà Wilson không mắc phải.

### Cách đọc kết quả cho đúng

Báo cáo giữ nguyên cam kết đã nêu ở phần kỳ vọng. Nếu phần lớn các cặp khung không khác biệt có ý
nghĩa thống kê, đó là **bằng chứng khẳng định luận điểm trung tâm** rằng nền toán học quyết định
kết quả còn khung thư viện chỉ là phương tiện biểu đạt, chứ không phải một thất bại của thực
nghiệm. Nếu có cặp nào khác biệt thật, con số đó vẫn được giữ nguyên trong tệp JSON và trong bảng,
và khác biệt ấy phải được quy cho một chi tiết hiện thực cụ thể, chẳng hạn thứ tự lô khác nhau,
cách khởi tạo mặc định khác nhau, hoặc thiết bị tính toán khác nhau, chứ không được quy cho một
khung "tốt hơn" khung kia một cách chung chung.

Giới hạn cần nói thẳng của thực nghiệm này: năm hạt giống là một cỡ mẫu nhỏ để ước lượng độ lệch
chuẩn, nên bản thân độ lệch chuẩn cũng mang sai số đáng kể. Con số năm được chọn vì ngân sách thời
gian, và việc nâng lên hai mươi hạt giống sẽ cho ước lượng phương sai chắc hơn. Ngoài ra, McNemar
và Wilson ở đây được tính trên dự đoán của hạt giống 42, đúng cấu hình đã công bố ở các chương
trước, nên hai phần đó trả lời câu hỏi về **chính các mô hình đã được báo cáo** chứ không phải về
toàn bộ họ mô hình sinh ra từ mọi hạt giống. Bảng độ bền qua năm hạt giống ở Phần B là bước bù đắp
một phần cho giới hạn đó.

Kết quả của notebook được ghi vào `../reports/metrics_statistical.json`, cùng ba hình
`fig_seed_variance.png`, `fig_mcnemar_matrix.png` và `fig_wilson_ci.png` trong
`../reports/figures/`.